In [ ]:
# SECTION 1: INSTALL DEPENDENCIES

print(" Installing dependencies...")
import subprocess, sys

packages = [
    "datasets", "transformers>=4.36.0", "sentence-transformers",
    "faiss-cpu", "torch", "torchvision",
    "Pillow", "tqdm", "matplotlib", "seaborn",
    "scikit-learn", "codecarbon",
    "numpy", "pandas", "openpyxl", "statsmodels",
    "accelerate>=0.26.0",
    "einops",
    "timm>=0.9.0",
    "torchvision>=0.15.0",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print(" All dependencies installed!\n")

In [ ]:
# SECTION 2: IMPORTS & SETUP

import os, random, warnings, datetime, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data_utils
from PIL import Image, ImageEnhance
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Any
from tqdm import tqdm
from collections import Counter
import logging
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

plt.rcParams.update({
    "figure.dpi":          150,
    "savefig.dpi":         300,
    "savefig.bbox":        "tight",
    "savefig.facecolor":   "white",
    "figure.facecolor":    "white",
    "axes.facecolor":      "#f8f9fa",
    "axes.grid":           True,
    "grid.alpha":          0.35,
    "grid.linestyle":      "--",
    "font.family":         "DejaVu Sans",
    "font.size":           11,
    "axes.titlesize":      13,
    "axes.titleweight":    "bold",
    "axes.labelsize":      11,
    "axes.labelweight":    "bold",
    "xtick.labelsize":     9,
    "ytick.labelsize":     9,
    "legend.fontsize":     9,
    "legend.framealpha":   0.85,
})

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    confusion_matrix, classification_report, f1_score,
    roc_curve, auc, roc_auc_score, accuracy_score,
    precision_recall_fscore_support, precision_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.decomposition import PCA
from codecarbon import EmissionsTracker
import torchvision.transforms as T
import torchvision.models as tv_models

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

try:
    from transformers import Blip2Processor, Blip2ForConditionalGeneration
    _BLIP2_AVAILABLE = True
except ImportError:
    _BLIP2_AVAILABLE = False
    logger.warning("  Blip2 not available — VLM disabled.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Device: {device}")

# SECTION 3: CONFIGURATION - UPDATED FOR BS/LB FOCUS

class Config:
    HF_DATASET_ID:   str = "Subh775/Rice-Disease-Augmented"
    HF_SPLIT:        str = "train"

    TOTAL_SAMPLE_SIZE: int   = 5000
    SAMPLES_PER_CLASS: int   = 0
    REMAINDER_CLASSES: int   = 0

    SMART_FARMING_CSV: str = "/kaggle/input/datasets/ytarunjitmeitei/iot-sensor/Smart_Farming_Crop_Yield_2024.csv"

    OUTPUT_DIR:  str = "/kaggle/working/outputs"
    PLOTS_DIR:   str = "/kaggle/working/outputs/plots"
    LOGS_DIR:    str = "/kaggle/working/outputs/logs"
    EMISSIONS_FILE: str = "/kaggle/working/outputs/emissions.csv"

    TRAIN_RATIO: float = 0.70
    VAL_RATIO:   float = 0.20
    TEST_RATIO:  float = 0.10
    TRAIN_SAMPLES: int = 0
    VAL_SAMPLES:   int = 0
    TEST_SAMPLES:  int = 0

    NUM_EVAL_CLASSES: int = 4

    CLIP_MODEL:    str   = "clip-ViT-B-32"
    EMBEDDING_DIM: int   = 512
    TOP_K_SIMILAR: int   = 11
    USE_PCA:       bool  = True
    PCA_DIM:       int   = 64

    TTA_VIEWS:             int   = 3
    QUERY_EMBED_NOISE_STD: float = 0.05

    SIM_THRESHOLD:         float = 0.68
    CONFUSION_BLEND_ALPHA: float = 0.15

    DEFAULT_LOCATION: str   = "Manipur, India"
    DEFAULT_LAT:      float = 24.65
    DEFAULT_LON:      float = 93.94

    # UPDATED: Enhanced parameters for BS/LB discrimination
    USE_CNN_CLASSIFIER:      bool  = True
    CNN_EPOCHS:              int   = 100
    CNN_LR:                  float = 2e-4
    CNN_BATCH_SIZE:          int   = 20
    CNN_DROPOUT:             float = 0.50
    CNN_IMG_SIZE:            int   = 300
    CNN_WEIGHT_DECAY:        float = 2e-4
    CNN_WARMUP_EPOCHS:       int   = 5
    CNN_MIXUP_ALPHA:         float = 0.0
    CNN_CUTMIX_ALPHA:        float = 0.4
    CNN_LABEL_SMOOTH:        float = 0.15
    CNN_FOCAL_GAMMA:         float = 2.5
    CNN_TWOHEAD_WEIGHT:      float = 0.35
    CNN_ENSEMBLE_WEIGHT:     float = 0.60
    RAG_ENSEMBLE_WEIGHT:     float = 0.40

    # UPDATED: Enhanced binary arbiter
    USE_BINARY_ARBITER:       bool  = True
    ARBITER_AMBIGUITY_THRESH: float = 0.40
    ARBITER_EPOCHS:           int   = 30
    ARBITER_LR:               float = 1e-4
    ARBITER_BATCH_SIZE:       int   = 16
    ARBITER_DROPOUT:          float = 0.40
    ARBITER_WEIGHT:           float = 0.65

    # UPDATED: Enhanced neural fusion
    USE_NEURAL_FUSION:       bool  = True
    NN_FUSION_EPOCHS:        int   = 25
    NN_FUSION_LR:            float = 3e-4
    NN_FUSION_BATCH_SIZE:    int   = 32
    NN_FUSION_MAX_SAMPLES:   int   = 8000
    NN_FUSION_DROPOUT:       float = 0.35

    VLM_MODEL_ID:       str   = "Salesforce/blip2-opt-2.7b"
    VLM_ENABLED:        bool  = True
    VLM_LOAD_IN_8BIT:   bool  = False
    VLM_LOAD_IN_4BIT:   bool  = False
    VLM_MAX_NEW_TOKENS: int   = 64
    VLM_RERANK_WEIGHT:  float = 0.30
    VLM_RERANK_TOPK:    int   = 4

    TOP_N_CROPS: int = 5

config = Config()

for _d in [config.OUTPUT_DIR, config.PLOTS_DIR, config.LOGS_DIR]:
    os.makedirs(_d, exist_ok=True)
print(f" Directories ready:\n"
      f"   {config.OUTPUT_DIR}\n"
      f"   {config.PLOTS_DIR}\n"
      f"   {config.LOGS_DIR}")

def _save(fig: plt.Figure, filename: str) -> str:
    path = os.path.join(config.PLOTS_DIR, filename)
    try:
        fig.savefig(path, dpi=300, bbox_inches="tight",
                    facecolor="white", edgecolor="none")
        print(f"  {path}")
    except Exception as exc:
        logger.error(f"Could not save {filename}: {exc}")
    finally:
        plt.close(fig)
    return path

# SECTION 4: LOAD DATASET + BALANCED SAMPLING + SPLIT

from datasets import load_dataset

print(f" Loading '{config.HF_DATASET_ID}' ...")
hf_ds = load_dataset(config.HF_DATASET_ID, split=config.HF_SPLIT)
print(f" {len(hf_ds):,} rows loaded")

label_feature = hf_ds.features["label"]
label_names: List[str] = (label_feature.names if hasattr(label_feature, "names")
                           else [str(i) for i in sorted(set(hf_ds["label"]))])

num_classes_total: int = len(label_names)
label2idx: Dict[str, int] = {n: i for i, n in enumerate(label_names)}

raw_class_counts: Dict[int, int] = {}
for lbl in hf_ds["label"]:
    raw_class_counts[lbl] = raw_class_counts.get(lbl, 0) + 1

print(f"\n {num_classes_total} classes:")
for i, n in enumerate(label_names):
    print(f"   [{i}] {n:<40} ({raw_class_counts.get(i, 0)})")

TOTAL_TARGET = config.TOTAL_SAMPLE_SIZE
base_quota   = TOTAL_TARGET // num_classes_total
remainder    = TOTAL_TARGET  % num_classes_total

sorted_by_count = sorted(raw_class_counts.items(),
                         key=lambda x: (x[1], x[0]), reverse=True)
bonus_classes   = {ci for ci, _ in sorted_by_count[:remainder]}

quota_per_class: Dict[int, int] = {
    ci: min(base_quota + (1 if ci in bonus_classes else 0),
            raw_class_counts.get(ci, 0))
    for ci in range(num_classes_total)
}
config.SAMPLES_PER_CLASS = base_quota
config.REMAINDER_CLASSES = remainder

print(f"\n Balanced sampling (target={TOTAL_TARGET}):")
for ci, q in quota_per_class.items():
    print(f"   [{ci}] {label_names[ci]:<40} → {q}")
print(f"   TOTAL: {sum(quota_per_class.values())}")

rng_split = random.Random(42)

all_class_indices: Dict[int, List[int]] = {i: [] for i in range(num_classes_total)}
for row_idx, lbl in enumerate(hf_ds["label"]):
    all_class_indices[lbl].append(row_idx)

balanced_class_indices: Dict[int, List[int]] = {}
for ci, idxs in all_class_indices.items():
    shuffled = list(idxs); rng_split.shuffle(shuffled)
    balanced_class_indices[ci] = shuffled[:quota_per_class[ci]]

train_samples: List[Tuple[int, int]] = []
val_samples:   List[Tuple[int, int]] = []
test_samples:  List[Tuple[int, int]] = []

for ci, idxs in balanced_class_indices.items():
    shuffled = list(idxs); rng_split.shuffle(shuffled)
    n      = len(shuffled)
    n_test = max(1, int(n * config.TEST_RATIO))
    n_val  = max(1, int(n * config.VAL_RATIO))
    test_samples.extend( [(i, ci) for i in shuffled[:n_test]] )
    val_samples.extend(  [(i, ci) for i in shuffled[n_test:n_test+n_val]] )
    train_samples.extend([(i, ci) for i in shuffled[n_test+n_val:]] )

rng_split.shuffle(train_samples)
rng_split.shuffle(val_samples)
rng_split.shuffle(test_samples)

config.TRAIN_SAMPLES = len(train_samples)
config.VAL_SAMPLES   = len(val_samples)
config.TEST_SAMPLES  = len(test_samples)
total_samples        = config.TRAIN_SAMPLES + config.VAL_SAMPLES + config.TEST_SAMPLES

print(f"\n Split → Train:{config.TRAIN_SAMPLES} | Val:{config.VAL_SAMPLES} | Test:{config.TEST_SAMPLES}")

val_row_ids  = {i for i, _ in val_samples}
test_row_ids = {i for i, _ in test_samples}
assert len(val_row_ids & test_row_ids) == 0, "❌ DATA LEAK!"
print(" Zero overlap confirmed.")

def plot_dataset_summary() -> None:
    cls_tr = Counter(lbl for _, lbl in train_samples)
    cls_va = Counter(lbl for _, lbl in val_samples)
    cls_te = Counter(lbl for _, lbl in test_samples)
    short  = [n.replace("Rice___", "").replace("_", " ") for n in label_names]
    x = np.arange(num_classes_total); w = 0.26
    PAL = ["#4f86c6", "#f5a623", "#7ed321"]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].bar(x-w, [cls_tr.get(i,0) for i in range(num_classes_total)],
                w, label="Train", color=PAL[0], edgecolor="white")
    axes[0].bar(x,   [cls_va.get(i,0) for i in range(num_classes_total)],
                w, label="Val",   color=PAL[1], edgecolor="white")
    axes[0].bar(x+w, [cls_te.get(i,0) for i in range(num_classes_total)],
                w, label="Test",  color=PAL[2], edgecolor="white")
    axes[0].set_xticks(x); axes[0].set_xticklabels(short, rotation=20, ha="right")
    axes[0].set_ylabel("Images"); axes[0].set_title("Per-Class Split Distribution")
    axes[0].legend()

    totals = [config.TRAIN_SAMPLES, config.VAL_SAMPLES, config.TEST_SAMPLES]
    lbls   = [f"Train\n{config.TRAIN_SAMPLES}",
              f"Val\n{config.VAL_SAMPLES}",
              f"Test\n{config.TEST_SAMPLES}"]
    axes[1].pie(totals, labels=lbls, colors=PAL, autopct="%1.1f%%",
                startangle=140, wedgeprops=dict(edgecolor="white", lw=2))
    axes[1].set_title(f"Dataset Split  (Total = {total_samples})")

    fig.suptitle("Balanced Dataset — Split Summary", fontsize=14)
    plt.tight_layout()
    _save(fig, "00_dataset_summary.png")

plot_dataset_summary()

# SECTION 5: EVAL CLASS SELECTION

ALLOWED_LABEL_INDICES: set = set(range(num_classes_total))

def select_eval_classes(n: int = 4) -> List[int]:
    counts = Counter(lbl for _, lbl in test_samples)
    top    = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:n]
    idxs   = sorted([lbl for lbl, _ in top])
    print(f"\n Eval classes: {[label_names[i] for i in idxs]}")
    return idxs

EVAL_CLASS_INDICES: List[int]     = select_eval_classes(config.NUM_EVAL_CLASSES)
EVAL_LABEL_NAMES:  List[str]      = [label_names[i] for i in EVAL_CLASS_INDICES]
ORIG_TO_EVAL:      Dict[int, int] = {orig: ei for ei, orig in enumerate(EVAL_CLASS_INDICES)}
SHORT_NAMES:       List[str]      = [n.replace("Rice___","").replace("_"," ")
                                     for n in EVAL_LABEL_NAMES]

# SECTION 6: DATA STRUCTURES

@dataclass
class PlantImageData:
    image_id:     str
    image_pil:    Image.Image
    label:        int
    label_name:   str
    disease_type: str
    plant_type:   str  = "Rice"
    is_healthy:   bool = False
    embedding:    Optional[np.ndarray] = None

@dataclass
class DiagnosisResult:
    disease:         str
    plant_species:   str
    confidence:      float
    is_healthy:      bool
    predicted_label: int = -1
    evidence:        List[str] = field(default_factory=list)
    agent_name:      str = ""

@dataclass
class CropRecommendation:
    crop:           str
    score:          float
    confidence_pct: float
    reasoning:      str
    soil_match:     str
    climate_match:  str
    season:         str

# SECTION 7: RICE DISEASE INFO + DATASET PROCESSOR

RICE_DISEASE_INFO: Dict[str, Dict] = {
    "rice___brown_spot":  {"desc": "Brown Spot (Bipolaris oryzae)",  "healthy": False},
    "rice___healthy":     {"desc": "Healthy Rice Leaf",               "healthy": True},
    "rice___leaf_blast":  {"desc": "Leaf Blast (Magnaporthe oryzae)", "healthy": False},
    "rice___neck_blast":  {"desc": "Neck Blast (Magnaporthe oryzae)", "healthy": False},
    "brown_spot":         {"desc": "Brown Spot (Bipolaris oryzae)",  "healthy": False},
    "healthy":            {"desc": "Healthy Rice Leaf",               "healthy": True},
    "leaf_blast":         {"desc": "Leaf Blast (Magnaporthe oryzae)", "healthy": False},
    "neck_blast":         {"desc": "Neck Blast (Magnaporthe oryzae)", "healthy": False},
}

def _lookup_disease_info(raw_label: str) -> Dict:
    key = raw_label.lower().replace(" ", "_").replace("-", "_")
    if key in RICE_DISEASE_INFO: return RICE_DISEASE_INFO[key]
    for k, v in RICE_DISEASE_INFO.items():
        if k in key or key in k: return v
    return {"desc":    raw_label.replace("_"," ").replace("Rice___","").title(),
            "healthy": any(w in key for w in ["healthy","normal","fresh"])}

class RiceLeafProcessor:
    def __init__(self):
        self.label_names = label_names
        self.mapping     = self._build_mapping()
        print(f"🌾 {len(self.label_names)} classes: {self.label_names}")

    def _build_mapping(self) -> Dict[str, Dict]:
        return {lbl: {"plant":"Rice","disease":_lookup_disease_info(lbl)["desc"],
                      "healthy":_lookup_disease_info(lbl)["healthy"],"raw_label":lbl}
                for lbl in self.label_names}

    def _load_row(self, row_idx: int) -> Optional[Image.Image]:
        try:
            row = hf_ds[row_idx]; img = row["image"]
            if isinstance(img, Image.Image): return img.convert("RGB")
            if isinstance(img, bytes):
                import io; return Image.open(io.BytesIO(img)).convert("RGB")
            return None
        except Exception as e:
            logger.debug(f"Row {row_idx}: {e}"); return None

    def load_split(self, split: str = "train",
                   max_samples: int = None,
                   filter_classes: Optional[set] = None) -> List[PlantImageData]:
        split_map = {"train": train_samples, "val": val_samples, "test": test_samples}
        samples   = split_map[split]
        if max_samples: samples = samples[:max_samples]
        items: List[PlantImageData] = []
        for ei, (row_idx, label) in enumerate(tqdm(samples, desc=f"Loading {split}")):
            if filter_classes is not None and label not in filter_classes: continue
            img = self._load_row(row_idx)
            if img is None: continue
            lname = self.label_names[label]; meta = self.mapping[lname]
            items.append(PlantImageData(
                image_id=f"{split}_{ei}_r{row_idx}", image_pil=img,
                label=label, label_name=lname,
                disease_type=meta["disease"], plant_type="Rice",
                is_healthy=meta["healthy"]))
        return items

processor = RiceLeafProcessor()

# SECTION 8: CLIP EMBEDDING GENERATOR

from sentence_transformers import SentenceTransformer

class CLIPEmbeddingGenerator:
    TTA_CONFIGS = [
        {},
        {"brightness": 1.25, "contrast": 1.20},
        {"brightness": 0.75, "contrast": 0.80},
        {"sharpness": 1.6,  "color": 1.30},
        {"sharpness": 0.5,  "color": 0.75, "brightness": 1.15},
    ]

    def __init__(self, model_name: str = config.CLIP_MODEL):
        print(f" Loading CLIP: {model_name}")
        self.model      = SentenceTransformer(model_name, device=device)
        self._raw_dim   = self.model.encode("test", convert_to_tensor=True).shape[0]
        self.pca        : Optional[PCA] = None
        self.pca_fitted : bool = False
        self.dim        = config.PCA_DIM if config.USE_PCA else self._raw_dim

    def fit_pca(self, index_images: List[PlantImageData]) -> None:
        if not config.USE_PCA: return
        print(f"\n Fitting PCA on {len(index_images)} val images...")
        raw = [self._encode_raw(it.image_pil, {})
               for it in tqdm(index_images, desc="  PCA fit")]
        X = np.stack(raw).astype(np.float32)
        self.pca = PCA(n_components=config.PCA_DIM, random_state=42)
        self.pca.fit(X)
        explained = float(self.pca.explained_variance_ratio_.sum())
        self.pca_fitted = True; self.dim = config.PCA_DIM
        print(f" PCA fitted | variance retained: {explained*100:.1f}%")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        cum = np.cumsum(self.pca.explained_variance_ratio_) * 100
        axes[0].plot(cum, color="#4f86c6", lw=2)
        axes[0].axhline(explained*100, color="crimson", ls="--",
                        label=f"Retained: {explained*100:.1f}%")
        axes[0].set_xlabel("Component"); axes[0].set_ylabel("Cumulative Var (%)")
        axes[0].set_title("PCA Cumulative Explained Variance"); axes[0].legend()

        top20 = self.pca.explained_variance_ratio_[:20] * 100
        axes[1].bar(range(1, len(top20)+1), top20, color="#7ed321", edgecolor="white")
        axes[1].set_xlabel("Component (top 20)"); axes[1].set_ylabel("Var (%)")
        axes[1].set_title("Top-20 Component Variance")

        fig.suptitle("CLIP + PCA Analysis", fontsize=14)
        plt.tight_layout()
        _save(fig, "01_pca_variance.png")

    def _encode_raw(self, img: Image.Image, cfg: dict) -> np.ndarray:
        if img.mode != "RGB": img = img.convert("RGB")
        for aug, cls_ in [("brightness", ImageEnhance.Brightness),
                           ("contrast",   ImageEnhance.Contrast),
                           ("sharpness",  ImageEnhance.Sharpness),
                           ("color",      ImageEnhance.Color)]:
            if aug in cfg: img = cls_(img).enhance(cfg[aug])
        if img.size != (224, 224): img = img.resize((224,224), Image.Resampling.LANCZOS)
        emb = self.model.encode(img, convert_to_tensor=True)
        emb = emb / emb.norm()
        return emb.cpu().numpy().astype(np.float32)

    def _apply_pca(self, emb: np.ndarray) -> np.ndarray:
        if not (config.USE_PCA and self.pca_fitted and self.pca is not None):
            return emb
        p = self.pca.transform(emb.reshape(1,-1))[0].astype(np.float32)
        return p / (np.linalg.norm(p) + 1e-8)

    def encode_image(self, img: Image.Image, tta_idx: int = 0,
                     add_noise: bool = False) -> np.ndarray:
        emb = self._apply_pca(self._encode_raw(img, self.TTA_CONFIGS[tta_idx % len(self.TTA_CONFIGS)]))
        if add_noise and config.QUERY_EMBED_NOISE_STD > 0:
            n = np.random.normal(0, config.QUERY_EMBED_NOISE_STD, emb.shape).astype(np.float32)
            emb = (emb + n); emb /= (np.linalg.norm(emb) + 1e-8)
        return emb

    def encode_text(self, text: str) -> np.ndarray:
        emb = self.model.encode(text, convert_to_tensor=True)
        emb = emb / emb.norm()
        return self._apply_pca(emb.cpu().numpy().astype(np.float32))

clip_generator = CLIPEmbeddingGenerator()

# SECTION 9: RAG 1 — FAISS VECTOR DATABASE

import faiss

class RiceVectorDB:
    def __init__(self):
        self.dim = None; self.data: List[PlantImageData] = []
        self.index = None; self.label_centroids: Dict[str, np.ndarray] = {}

    def add(self, image_data: List[PlantImageData], embedder: CLIPEmbeddingGenerator):
        self.dim = embedder.dim; self.index = faiss.IndexFlatIP(self.dim)
        batch_embs: List[np.ndarray] = []; emb_map: Dict[str, List] = {}
        for item in tqdm(image_data, desc="RAG 1: Indexing"):
            emb = embedder.encode_image(item.image_pil, tta_idx=0)
            item.embedding = emb; batch_embs.append(emb); self.data.append(item)
            emb_map.setdefault(item.label_name, []).append(emb)
        embs = np.array(batch_embs, dtype=np.float32); faiss.normalize_L2(embs)
        self.index.add(embs)
        for lbl, vecs in emb_map.items():
            c = np.mean(np.stack(vecs), axis=0); c /= (np.linalg.norm(c)+1e-8)
            self.label_centroids[lbl] = c.astype(np.float32)
        print(f"RAG 1: {len(batch_embs)} indexed | dim={self.dim} | {len(self.label_centroids)} centroids")

    def search(self, query_emb: np.ndarray, k: int = 10) -> List[Tuple[PlantImageData, float]]:
        q = query_emb.reshape(1,-1).astype(np.float32); faiss.normalize_L2(q)
        sims, idxs = self.index.search(q, k)
        return [(self.data[i], float(sims[0][j])) for j,i in enumerate(idxs[0]) if i < len(self.data)]

    def centroid_scores(self, query_emb: np.ndarray) -> Dict[str, float]:
        q = query_emb / (np.linalg.norm(query_emb)+1e-8)
        return {lbl: float(np.dot(q,c)) for lbl,c in self.label_centroids.items()}

vector_db = RiceVectorDB()

# SECTION 10: RAG 2 — CROP RECOMMENDATION ENGINE

class CropRecommendationRAG:
    COL_ALIASES = {
        "crop":        ["crop","crop_type","crop_name","label","plant","recommended_crop","Crop"],
        "temperature": ["temperature","temp","temperature_c","avg_temp","Temperature"],
        "humidity":    ["humidity","humidity_pct","relative_humidity","Humidity"],
        "rainfall":    ["rainfall","precipitation","annual_rainfall","Rainfall"],
        "ph":          ["ph","soil_ph","ph_value","pH"],
        "nitrogen":    ["nitrogen","n","n_content","Nitrogen","N"],
        "phosphorus":  ["phosphorus","p","p_content","Phosphorus","P"],
        "potassium":   ["potassium","k","k_content","Potassium","K"],
        "soil_type":   ["soil_type","soil","soil_name","Soil_Type"],
        "region":      ["region","location","state","district","Region","Location","State"],
        "season":      ["season","Season","growing_season"],
        "yield":       ["yield","crop_yield","production","Yield","Crop_Yield_MT_per_HA"],
        "latitude":    ["latitude","lat","Latitude"],
        "longitude":   ["longitude","lon","long","Longitude"],
    }
    NUMERIC_FEATURES  = ["temperature","humidity","rainfall","ph","nitrogen","phosphorus","potassium"]
    CATEGORICAL_FEATS = ["soil_type","season","region"]

    def __init__(self, csv_path: str):
        self.csv_path = csv_path; self.df_raw = None; self.col_map: Dict[str,str] = {}
        self.feature_cols: List[str] = []; self.target_col: str = ""
        self.rf_model = None; self.gb_model = None
        self.label_encoder = LabelEncoder(); self.scaler = StandardScaler()
        self.cat_encoders: Dict[str,LabelEncoder] = {}
        self.crop_profiles: Dict[str,Dict] = {}; self.unique_crops: List[str] = []
        self.faiss_index = None; self.faiss_records: List[Dict] = []
        self.feature_dim: int = 0; self.rf_cv_score: float = 0.0
        self._load_and_train()

    def _detect_col(self, df, canonical):
        aliases = self.COL_ALIASES.get(canonical, [canonical])
        clow    = {c.lower().strip(): c for c in df.columns}
        for a in aliases:
            if a.lower() in clow: return clow[a.lower()]
        return None

    def _build_col_map(self, df):
        for c in self.COL_ALIASES:
            a = self._detect_col(df, c)
            if a: self.col_map[c] = a

    def _load_and_train(self):
        print(f"\n RAG 2: Loading {self.csv_path}")
        try:
            df = pd.read_csv(self.csv_path)
            print(f"   Shape: {df.shape}")
        except Exception as e:
            print(f"  CSV failed ({e}). Heuristic fallback.")
            self._build_heuristic_profiles(); return

        self._build_col_map(df)
        target = self.col_map.get("crop") or (df.select_dtypes("object").columns[0]
                                               if len(df.select_dtypes("object").columns) else None)
        if target is None:
            self._build_heuristic_profiles(); return

        self.target_col = target
        df[target] = df[target].astype(str).str.strip().str.lower()
        df = df[df[target].notna() & (df[target] != "nan")]

        num_feat_cols = []
        for c in self.NUMERIC_FEATURES:
            a = self.col_map.get(c)
            if a and a in df.columns:
                df[a] = pd.to_numeric(df[a], errors="coerce"); num_feat_cols.append(a)

        cat_feat_cols = []
        for c in self.CATEGORICAL_FEATS:
            a = self.col_map.get(c)
            if a and a in df.columns and a != target:
                df[a] = df[a].astype(str).str.strip().str.lower()
                le = LabelEncoder(); df[a+"_enc"] = le.fit_transform(df[a].fillna("unknown"))
                self.cat_encoders[a] = le; cat_feat_cols.append(a+"_enc")

        all_feat_cols = num_feat_cols + cat_feat_cols
        if not all_feat_cols: self._build_heuristic_profiles(); return

        df_clean = df.dropna(subset=num_feat_cols+[target]).reset_index(drop=True)
        if len(df_clean) < 20: self._build_heuristic_profiles(); return

        self.df_raw = df_clean; self.feature_cols = all_feat_cols
        y = self.label_encoder.fit_transform(df_clean[target])
        self.unique_crops = list(self.label_encoder.classes_)

        X_num = df_clean[num_feat_cols].fillna(df_clean[num_feat_cols].median()).values
        X_cat = df_clean[cat_feat_cols].values if cat_feat_cols else np.zeros((len(df_clean),0))
        X = np.hstack([X_num, X_cat]).astype(np.float32)
        X_sc = self.scaler.fit_transform(X); self.feature_dim = X_sc.shape[1]

        print("   Training RandomForest...")
        self.rf_model = RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                               random_state=42, n_jobs=-1)
        self.rf_model.fit(X_sc, y)
        self.rf_cv_score = cross_val_score(self.rf_model, X_sc, y, cv=5, scoring="accuracy").mean()
        print(f" RF CV: {self.rf_cv_score*100:.1f}%")

        print(" Training GradientBoosting...")
        self.gb_model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                                   max_depth=5, random_state=42)
        self.gb_model.fit(X_sc, y)

        self.faiss_index = faiss.IndexFlatL2(self.feature_dim)
        X_f32 = X_sc.astype(np.float32); self.faiss_index.add(X_f32)
        self.faiss_records = [{"crop": self.unique_crops[y[i]], "features": X_f32[i]}
                               for i in range(len(df_clean))]
        self._build_crop_profiles(df_clean, num_feat_cols, target)

        if hasattr(self.rf_model, "feature_importances_") and all_feat_cols:
            imp   = self.rf_model.feature_importances_
            short = [c.replace("_enc","").replace("Temperature","Temp")
                      .replace("Humidity","Hum").replace("Rainfall","Rain")
                     for c in all_feat_cols]
            order = np.argsort(imp)[::-1]
            fig, ax = plt.subplots(figsize=(10, 4))
            ax.bar(range(len(short)), imp[order], color="#4f86c6", edgecolor="white")
            ax.set_xticks(range(len(short)))
            ax.set_xticklabels([short[i] for i in order], rotation=40, ha="right")
            ax.set_ylabel("Importance"); ax.set_title("RandomForest Feature Importances")
            plt.tight_layout(); _save(fig, "02_rf_feature_importance.png")

        print(f" RAG 2 ready | {len(self.unique_crops)} crops")

    def _build_crop_profiles(self, df, num_cols, target_col):
        for crop, grp in df.groupby(target_col):
            p: Dict[str,Any] = {"count": len(grp)}
            for col in num_cols:
                p[f"avg_{col}"] = float(grp[col].mean())
                p[f"std_{col}"] = float(grp[col].std())
            sc = self.col_map.get("season")
            if sc and sc in grp.columns:
                sv = grp[sc].value_counts()
                p["best_season"] = sv.index[0] if len(sv) else "unknown"
            st = self.col_map.get("soil_type")
            if st and st in grp.columns:
                sv = grp[st].value_counts()
                p["best_soil"] = sv.index[0] if len(sv) else "unknown"
            self.crop_profiles[str(crop)] = p

    def _build_heuristic_profiles(self):
        self.unique_crops = ["rice","wheat","maize","sugarcane","jute","cotton",
                             "soybean","groundnut","mungbean","lentil",
                             "potato","tomato","onion","chilli","turmeric"]
        self.crop_profiles = {
            "rice":      {"best_season":"kharif","best_soil":"clay loam","avg_temperature":26,"avg_rainfall":1200,"avg_ph":6.0},
            "wheat":     {"best_season":"rabi",  "best_soil":"loam",     "avg_temperature":15,"avg_rainfall":400, "avg_ph":6.5},
            "maize":     {"best_season":"kharif","best_soil":"loam",     "avg_temperature":22,"avg_rainfall":600, "avg_ph":6.2},
            "sugarcane": {"best_season":"annual","best_soil":"sandy loam","avg_temperature":27,"avg_rainfall":1500,"avg_ph":6.5},
            "jute":      {"best_season":"kharif","best_soil":"alluvial", "avg_temperature":28,"avg_rainfall":1800,"avg_ph":7.0},
            "cotton":    {"best_season":"kharif","best_soil":"black cotton","avg_temperature":30,"avg_rainfall":600,"avg_ph":7.0},
            "soybean":   {"best_season":"kharif","best_soil":"loam",     "avg_temperature":25,"avg_rainfall":700, "avg_ph":6.5},
            "groundnut": {"best_season":"kharif","best_soil":"sandy loam","avg_temperature":28,"avg_rainfall":500, "avg_ph":6.0},
            "mungbean":  {"best_season":"kharif","best_soil":"sandy loam","avg_temperature":28,"avg_rainfall":400, "avg_ph":6.5},
            "lentil":    {"best_season":"rabi",  "best_soil":"loam",     "avg_temperature":18,"avg_rainfall":300, "avg_ph":7.0},
            "potato":    {"best_season":"rabi",  "best_soil":"sandy loam","avg_temperature":18,"avg_rainfall":400, "avg_ph":5.5},
            "tomato":    {"best_season":"rabi",  "best_soil":"loam",     "avg_temperature":22,"avg_rainfall":400, "avg_ph":6.0},
            "onion":     {"best_season":"rabi",  "best_soil":"sandy loam","avg_temperature":20,"avg_rainfall":300, "avg_ph":6.5},
            "chilli":    {"best_season":"kharif","best_soil":"sandy loam","avg_temperature":26,"avg_rainfall":600, "avg_ph":6.0},
            "turmeric":  {"best_season":"kharif","best_soil":"loam",     "avg_temperature":25,"avg_rainfall":1500,"avg_ph":5.5},
        }
        self.feature_cols = []; print(" Heuristic profiles built.")

    def _build_query_vector(self, query):
        if not self.feature_cols or self.scaler is None: return None
        nv, cv = [], []
        for c in self.NUMERIC_FEATURES:
            a = self.col_map.get(c)
            if a and a in self.feature_cols: nv.append(float(query.get(c, 0.0)))
        for c in self.CATEGORICAL_FEATS:
            a = self.col_map.get(c); ec = (a+"_enc") if a else None
            if ec and ec in self.feature_cols:
                le = self.cat_encoders.get(a); raw = str(query.get(c,"unknown")).lower()
                try: cv.append(float(le.transform([raw])[0]))
                except: cv.append(0.0)
        if not nv and not cv: return None
        x = np.array(nv+cv, dtype=np.float32).reshape(1,-1)
        if x.shape[1] != self.feature_dim:
            xf = np.zeros((1, self.feature_dim), dtype=np.float32)
            w  = min(x.shape[1], self.feature_dim); xf[0,:w] = x[0,:w]; x = xf
        return self.scaler.transform(x).astype(np.float32)

    def recommend(self, query, top_n=None):
        top_n = top_n or config.TOP_N_CROPS
        q_vec = self._build_query_vector(query); ml_probs = None

        if q_vec is not None and self.rf_model is not None:
            ml_probs = 0.60*self.rf_model.predict_proba(q_vec)[0] + \
                       0.40*self.gb_model.predict_proba(q_vec)[0]

        rag_votes: Dict[str,float] = {}
        if q_vec is not None and self.faiss_index is not None:
            k_ret = min(20, len(self.faiss_records)); D, I = self.faiss_index.search(q_vec, k_ret)
            for rank, (dist, idx) in enumerate(zip(D[0], I[0])):
                if idx < 0: continue
                crop = self.faiss_records[idx]["crop"]
                w    = 1.0/(1.0+float(dist)) * np.exp(-rank*0.15)
                rag_votes[crop] = rag_votes.get(crop,0.0) + w
            tv = sum(rag_votes.values()) or 1.0
            rag_votes = {c: v/tv for c,v in rag_votes.items()}

        if ml_probs is not None:
            scores = {crop: 0.70*float(ml_probs[i]) + 0.30*rag_votes.get(crop,0.0)
                      for i,crop in enumerate(self.unique_crops)}
        elif rag_votes: scores = dict(rag_votes)
        else:
            qt = float(query.get("temperature",25.0))
            scores = {c: max(0.0, 1.0-abs(qt-p.get("avg_temperature",25.0))/30.0)
                      for c,p in self.crop_profiles.items()}

        sorted_crops = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
        total_score  = sum(s for _,s in sorted_crops) or 1.0
        recs = []
        for rank, (crop, score) in enumerate(sorted_crops):
            prof = self.crop_profiles.get(crop, {})
            recs.append(CropRecommendation(
                crop=crop.title(), score=round(score,4),
                confidence_pct=round(score/total_score*100,1),
                reasoning=self._reasoning(crop, rank+1, score, query, prof),
                soil_match=str(prof.get("best_soil",query.get("soil_type","loam"))).title(),
                climate_match=self._climate_match_desc(query, prof),
                season=str(prof.get("best_season",query.get("season","kharif"))).title()))
        return recs

    def _climate_match_desc(self, query, prof):
        parts = []
        if "temperature" in query:
            d = abs(float(query["temperature"]) - float(prof.get("avg_temperature", query["temperature"])))
            parts.append("Excellent temp" if d<3 else "Good temp" if d<7 else "Moderate temp" if d<12 else "Poor temp")
        if "rainfall" in query:
            ideal = float(prof.get("avg_rainfall", query["rainfall"]))
            d = abs(float(query["rainfall"]) - ideal) / (ideal+1)
            parts.append("Excellent rainfall" if d<0.15 else "Good rainfall" if d<0.30
                         else "Moderate rainfall" if d<0.50 else "Low rainfall")
        return ", ".join(parts) if parts else "N/A"

    def _reasoning(self, crop, rank, score, query, prof):
        parts = [f"Rank #{rank} (score={score:.4f})."]
        if "temperature" in query: parts.append(f"Temp {query['temperature']}°C vs ~{prof.get('avg_temperature','?')}°C.")
        if "rainfall"    in query: parts.append(f"Rain {query['rainfall']}mm vs ~{prof.get('avg_rainfall','?')}mm.")
        if prof.get("best_season"): parts.append(f"Season: {prof['best_season']}.")
        return " ".join(parts)

    def plot_recommendations(self, recs, location="", filename="11_crop_recommendations.png"):
        if not recs: return
        crops  = [r.crop for r in recs]; scores = [r.score for r in recs]
        confs  = [r.confidence_pct for r in recs]
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        clrs = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(crops)))
        bars = axes[0].barh(range(len(crops)), scores, color=clrs, edgecolor="white")
        for bar, sc in zip(bars, scores):
            axes[0].text(bar.get_width() + max(scores)*0.01,
                         bar.get_y()+bar.get_height()/2,
                         f"{sc:.4f}", va="center", fontsize=9, fontweight="bold")
        axes[0].set_yticks(range(len(crops))); axes[0].set_yticklabels(crops)
        axes[0].set_xlabel("Blended ML + RAG Score")
        axes[0].set_title(f"Crop Recommendations — {location}" if location else "Crop Recommendations")
        axes[0].set_xlim(0, max(scores)*1.25); axes[0].grid(axis="x", alpha=0.3)

        axes[1].pie(confs, labels=crops, autopct="%1.1f%%",
                    colors=plt.cm.tab10(np.linspace(0,1,len(crops))),
                    startangle=140, wedgeprops=dict(edgecolor="white", lw=1.5))
        axes[1].set_title("Confidence Distribution (%)")

        fig.suptitle("Agent 2 — Crop Recommendation (RAG 2)", fontsize=14)
        plt.tight_layout(); _save(fig, filename)

    def plot_crop_condition_heatmap(self, filename="12_crop_condition_heatmap.png"):
        if not self.crop_profiles: return
        keys = ["avg_temperature","avg_humidity","avg_rainfall",
                "avg_ph","avg_nitrogen","avg_phosphorus","avg_potassium"]
        pkeys = [k for k in keys if any(k in v for v in self.crop_profiles.values())]
        if not pkeys: return
        crops = list(self.crop_profiles.keys())[:20]
        mat   = np.full((len(crops), len(pkeys)), np.nan)
        for i, crop in enumerate(crops):
            for j, key in enumerate(pkeys):
                v = self.crop_profiles[crop].get(key)
                if v is not None: mat[i,j] = float(v)
        mn = np.nanmin(mat, axis=0); mx = np.nanmax(mat, axis=0)
        mat_n = (mat - mn) / (mx - mn + 1e-8)
        col_lbl = [k.replace("avg_","").replace("_"," ").title() for k in pkeys]

        fig, ax = plt.subplots(figsize=(max(10, len(pkeys)*1.5), max(6, len(crops)*0.45)))
        sns.heatmap(mat_n, annot=True, fmt=".2f", cmap="YlOrRd",
                    xticklabels=col_lbl, yticklabels=crops,
                    linewidths=0.4, ax=ax, cbar_kws={"label":"Normalised Value"})
        ax.set_title("Crop Growing Conditions Heatmap (Normalised)", fontsize=13)
        plt.tight_layout(); _save(fig, filename)

rag2_crop_recommender = CropRecommendationRAG(config.SMART_FARMING_CSV)

# SECTION 11: FIT PCA + BUILD RAG 1 FAISS INDEX

print(f"\n PCA fitting on VAL ({config.VAL_SAMPLES} imgs)...")
pca_fit_images = processor.load_split("val", filter_classes=ALLOWED_LABEL_INDICES)
clip_generator.fit_pca(pca_fit_images)
print(f"\n Building FAISS index from TRAIN ({config.TRAIN_SAMPLES} imgs)...")
index_images = processor.load_split("train", filter_classes=ALLOWED_LABEL_INDICES)
vector_db.add(index_images, clip_generator)
print(f"   FAISS: {len(index_images)} imgs indexed")

#SECTION 12: RAG CORE


class RiceDiseaseRAG:
    def __init__(self, vdb, clip_gen):
        self.vector_db = vdb; self.clip_gen = clip_gen

    def find_similar_cases(self, image_pil, k=config.TOP_K_SIMILAR, is_query=True):
        pool: Dict[str,Dict] = {}; n_views = config.TTA_VIEWS if is_query else 1
        for vi in range(n_views):
            emb = self.clip_gen.encode_image(image_pil, tta_idx=vi,
                                             add_noise=(is_query and vi>0))
            for item, sim in self.vector_db.search(emb, k=k*3):
                if item.image_id not in pool: pool[item.image_id] = {"item":item,"scores":[]}
                pool[item.image_id]["scores"].append(sim)
        agg = sorted([(d["item"], float(np.mean(d["scores"]))) for d in pool.values()],
                     key=lambda x: x[1], reverse=True)[:k*2]
        pe  = self.clip_gen.encode_image(image_pil, tta_idx=0)
        cs  = self.vector_db.centroid_scores(pe)
        return sorted([(it, 0.70*s + 0.30*cs.get(it.label_name,0.0)) for it,s in agg],
                      key=lambda x: x[1], reverse=True)[:k]

    def analyze_similar_cases(self, cases) -> Dict:
        if not cases:
            return {"top_diseases":[],"label_votes":{},"total_cases":0,
                    "avg_similarity":0.0,"is_healthy":False,"in_confusion_zone":False}
        ds: Dict[str,float] = {}; lv: Dict[str,float] = {}; hs = 0.0
        ts = sum(s for _,s in cases)
        for idx, (item, sim) in enumerate(cases):
            pw = np.exp(-idx*0.25); wsi = sim*pw; wt = wsi/(ts+1e-8)
            ds[item.disease_type] = ds.get(item.disease_type,0) + wsi
            lv[item.label_name]   = lv.get(item.label_name,0) + wt
            if item.is_healthy: hs += wt
        top_sim = cases[0][1]; inz = top_sim < config.SIM_THRESHOLD
        if inz and len(lv) >= 2:
            sv = sorted(lv.items(), key=lambda x: x[1], reverse=True)
            bl,bv = sv[0]; sl,sv2 = sv[1]
            if (bv - sv2) < 0.10:
                tr = bv * config.CONFUSION_BLEND_ALPHA
                lv[bl] = bv-tr; lv[sl] = sv2+tr
        td = sum(ds.values())
        return {"top_diseases":[{"disease":d,"score":s,"confidence":s/td}
                                for d,s in sorted(ds.items(),key=lambda x:x[1],reverse=True)[:3]],
                "label_votes":lv,"total_cases":len(cases),"avg_similarity":ts/len(cases),
                "is_healthy":hs>0.5,"in_confusion_zone":inz,"top_similarity":top_sim}

rag_system = RiceDiseaseRAG(vector_db, clip_generator)

# SECTION 13: BLIP-2 VLM RE-RANKER

class VLMDiseaseReranker:
    DISEASE_PROMPTS = {
        "Brown Spot (Bipolaris oryzae)":  ("Does this rice leaf have oval brown spots with yellow halos indicating brown spot disease? Answer yes or no."),
        "Leaf Blast (Magnaporthe oryzae)": ("Does this rice leaf show diamond-shaped lesions with grey centres and brown borders indicating leaf blast? Answer yes or no."),
        "Neck Blast (Magnaporthe oryzae)": ("Does this rice panicle show browning or rotting indicating neck blast disease? Answer yes or no."),
        "Healthy Rice Leaf":               ("Is this rice leaf healthy, uniformly green, free from spots or lesions? Answer yes or no."),
    }
    GENERIC = "Does this rice leaf show signs of {disease_name}? Answer yes or no."

    def __init__(self):
        self.processor=None; self.model=None; self.loaded=False
        self._yes_ids: List[int]=[]; self._no_ids: List[int]=[]

    def load(self) -> bool:
        if self.loaded: return True
        if not config.VLM_ENABLED or not _BLIP2_AVAILABLE: return False
        print(f"\n Loading BLIP-2: {config.VLM_MODEL_ID}")
        try:
            quant = {}
            if config.VLM_LOAD_IN_8BIT:
                from transformers import BitsAndBytesConfig; quant["quantization_config"]=BitsAndBytesConfig(load_in_8bit=True)
            elif config.VLM_LOAD_IN_4BIT:
                from transformers import BitsAndBytesConfig; quant["quantization_config"]=BitsAndBytesConfig(load_in_4bit=True)
            self.processor = Blip2Processor.from_pretrained(config.VLM_MODEL_ID)
            self.model = Blip2ForConditionalGeneration.from_pretrained(
                config.VLM_MODEL_ID,
                torch_dtype=torch.float16 if device=="cuda" else torch.float32,
                device_map="auto", **quant)
            self.model.eval()
            for w in ["yes","Yes"," yes"," Yes"]:
                self._yes_ids.extend(self.processor.tokenizer.encode(w,add_special_tokens=False))
            for w in ["no","No"," no"," No"]:
                self._no_ids.extend(self.processor.tokenizer.encode(w,add_special_tokens=False))
            self._yes_ids = list(set(self._yes_ids)); self._no_ids = list(set(self._no_ids))
            self.loaded = True
            print(f" BLIP-2 loaded on {next(self.model.parameters()).device}")
            return True
        except Exception as e:
            logger.error(f" VLM load failed: {e}"); return False

    @torch.no_grad()
    def score_label(self, image_pil, disease_name) -> float:
        if not self.loaded or self.model is None: return 0.5
        try:
            prompt  = self.DISEASE_PROMPTS.get(disease_name, self.GENERIC.format(disease_name=disease_name))
            inputs  = self.processor(images=image_pil, text=prompt, return_tensors="pt").to(
                self.model.device, torch.float16 if device=="cuda" else torch.float32)
            outputs = self.model(**inputs, decoder_input_ids=torch.tensor(
                [[self.model.config.text_config.bos_token_id]], device=self.model.device))
            probs   = torch.softmax(outputs.logits[0,-1,:].float(), dim=0)
            yp = sum(float(probs[t]) for t in self._yes_ids if t<len(probs))
            np_ = sum(float(probs[t]) for t in self._no_ids  if t<len(probs))
            tot = yp + np_; return (yp/tot) if tot>1e-9 else 0.5
        except Exception as e:
            logger.warning(f"VLM score failed: {e}"); return 0.5

    def rerank(self, image_pil, faiss_label_scores, processor_mapping, label_names_list):
        if not self.loaded or not faiss_label_scores: return None
        top = sorted(faiss_label_scores.items(), key=lambda x: x[1], reverse=True)[:config.VLM_RERANK_TOPK]
        fv  = np.array([s for _,s in top], dtype=np.float32)
        fn  = ((fv-fv.min())/(fv.max()-fv.min()+1e-8)).tolist()
        wv, wf = config.VLM_RERANK_WEIGHT, 1.0-config.VLM_RERANK_WEIGHT
        blended: Dict[str,float] = {}; evidence: List[str] = []
        for (lbl,_), fn_i in zip(top, fn):
            meta = processor_mapping.get(lbl, {}); vs = self.score_label(image_pil, meta.get("disease",lbl))
            b = wf*fn_i + wv*vs; blended[lbl] = b
            evidence.append(f"VLM '{lbl}': F={fn_i:.3f} V={vs:.3f} b={b:.3f}")
        best = max(blended, key=blended.__getitem__); meta = processor_mapping.get(best,{})
        pidx = label_names_list.index(best) if best in label_names_list else -1
        evidence.insert(0, f"VLM winner='{best}' score={blended[best]:.3f}")
        return DiagnosisResult(disease=meta.get("disease",best), plant_species="Rice",
                               confidence=float(np.clip(blended[best],0,1)),
                               is_healthy=meta.get("healthy",False), predicted_label=pidx,
                               evidence=evidence, agent_name="VLMReranker(BLIP-2)")

vlm_reranker = VLMDiseaseReranker()

# SECTION 13.1-A: UPDATED EFFICIENTNET-B5 CLASSIFIER WITH ENHANCED BS/LB FOCUS

import timm

def _make_train_transform(img_size: int = 300) -> T.Compose:
    """
    UPDATED: More aggressive augmentation targeting Brown Spot / Leaf Blast discrimination
    """
    return T.Compose([
        T.Resize((img_size + 48, img_size + 48)),
        T.RandomCrop(img_size),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.4),
        T.RandomRotation(degrees=45),
        T.ColorJitter(brightness=0.45, contrast=0.45, saturation=0.40, hue=0.15),
        T.RandomPerspective(distortion_scale=0.25, p=0.5),
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        T.RandomErasing(p=0.35, scale=(0.02, 0.15), ratio=(0.3, 3.3)),
    ])

def _make_val_transform(img_size: int = 300) -> T.Compose:
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

def _make_tta_transforms(img_size: int = 300) -> List[T.Compose]:
    base = [T.Resize((img_size, img_size)), T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]
    return [
        T.Compose(base),
        T.Compose([T.Resize((img_size+16, img_size+16)), T.CenterCrop(img_size)] + base[1:]),
        T.Compose([T.Resize((img_size, img_size)), T.RandomHorizontalFlip(p=1.0)] + base[1:]),
        T.Compose([T.Resize((img_size, img_size)),
                   T.ColorJitter(brightness=0.25, contrast=0.25)] + base[1:]),
        T.Compose([T.Resize((img_size, img_size)),
                   T.ColorJitter(saturation=0.30, hue=0.08)] + base[1:]),
    ]

class RiceImageDataset(data_utils.Dataset):
    def __init__(self, items: List["PlantImageData"],
                 label_map: Dict[int, int],
                 transform: T.Compose):
        self.items = items; self.label_map = label_map; self.transform = transform

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        img  = item.image_pil
        if img.mode != "RGB": img = img.convert("RGB")
        tensor = self.transform(img)
        label  = self.label_map[item.label]
        return tensor, label

def _cutmix_batch(x: torch.Tensor, y: torch.Tensor, alpha: float = 0.4):
    if alpha <= 0:
        lam = 1.0
        idx = torch.arange(x.size(0), device=x.device)
        return x, y, y[idx], lam
    lam   = float(np.random.beta(alpha, alpha))
    B, C, H, W = x.shape
    idx   = torch.randperm(B, device=x.device)
    cut_ratio = np.sqrt(1.0 - lam)
    cut_h = int(H * cut_ratio); cut_w = int(W * cut_ratio)
    cx = np.random.randint(W); cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W); x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H); y2 = np.clip(cy + cut_h // 2, 0, H)
    x_mix = x.clone()
    x_mix[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam_adj = 1.0 - float((x2-x1)*(y2-y1)) / float(H * W)
    return x_mix, y, y[idx], lam_adj

class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.5,
                 weight: Optional[torch.Tensor] = None,
                 label_smoothing: float = 0.15):
        super().__init__()
        self.gamma          = gamma
        self.weight         = weight
        self.label_smoothing = label_smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        nc = logits.size(1)
        with torch.no_grad():
            smooth_targets = torch.full_like(logits, self.label_smoothing / nc)
            smooth_targets.scatter_(1, targets.unsqueeze(1),
                                    1.0 - self.label_smoothing + self.label_smoothing / nc)
        log_p  = F.log_softmax(logits, dim=1)
        p      = torch.exp(log_p)
        focal  = (1 - p) ** self.gamma
        loss_n = -(smooth_targets * focal * log_p).sum(dim=1)
        if self.weight is not None:
            w = self.weight[targets]
            loss_n = loss_n * w
        return loss_n.mean()

class EfficientNetDiseaseClassifier(nn.Module):
    """
    UPDATED: Enhanced attention mechanism for BS/LB discrimination
    """
    def __init__(self, num_classes: int, dropout: float = 0.50,
                 bs_idx: int = 0, lb_idx: int = 2):
        super().__init__()
        self.bs_idx = bs_idx
        self.lb_idx = lb_idx
        self.backbone = timm.create_model(
            "efficientnet_b5", pretrained=True, num_classes=0,
            global_pool="avg", drop_rate=dropout)
        feat_dim = self.backbone.num_features
        
        self.se_gate = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 8),
            nn.ReLU(),
            nn.Linear(feat_dim // 8, feat_dim // 8),
            nn.ReLU(),
            nn.Linear(feat_dim // 8, feat_dim),
            nn.Sigmoid())
        
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, num_classes))
        
        self.aux_head = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout * 0.6),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout * 0.4),
            nn.Linear(256, 2))
        
        for m in list(self.head.modules()) + list(self.aux_head.modules()):
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor):
        feats = self.backbone(x)
        gate  = self.se_gate(feats)
        feats = feats * gate
        main_out = self.head(feats)
        aux_out  = self.aux_head(feats)
        return main_out, aux_out

    def predict(self, x: torch.Tensor) -> torch.Tensor:
        main_out, _ = self.forward(x)
        return main_out

def train_efficientnet_classifier(
        train_items: List["PlantImageData"],
        val_items:   List["PlantImageData"],
        eval_class_indices: List[int],
        label_names_list:   List[str],
        orig_to_eval:       Dict[int, int],
        model_tag: str = "main") -> Optional[EfficientNetDiseaseClassifier]:

    nc  = len(eval_class_indices)
    dev = torch.device(device)

    tr = [it for it in train_items if it.label in orig_to_eval]
    vl = [it for it in val_items   if it.label in orig_to_eval]
    if not tr:
        print(f" CNN {model_tag}: no train samples, skipping."); return None

    img_sz = config.CNN_IMG_SIZE
    tr_tf  = _make_train_transform(img_sz)
    vl_tf  = _make_val_transform(img_sz)

    tr_ds  = RiceImageDataset(tr, orig_to_eval, tr_tf)
    vl_ds  = RiceImageDataset(vl, orig_to_eval, vl_tf)
    
    # UPDATED: BS/LB focused sampling
    BS_ORIG_IDX = next((i for i,n in enumerate(label_names_list)
                        if "brown" in n.lower() and "spot" in n.lower()), None)
    LB_ORIG_IDX = next((i for i,n in enumerate(label_names_list)
                        if "leaf" in n.lower() and "blast" in n.lower()), None)
    
    bs_lb_indices = [i for i, it in enumerate(tr) 
                     if it.label in [BS_ORIG_IDX, LB_ORIG_IDX]]
    if len(bs_lb_indices) > 10:
        weights = [2.5 if i in bs_lb_indices else 1.0 for i in range(len(tr))]
        sampler = data_utils.WeightedRandomSampler(weights=weights, num_samples=len(tr), replacement=True)
        tr_ld = data_utils.DataLoader(tr_ds, batch_size=config.CNN_BATCH_SIZE,
                                      sampler=sampler, num_workers=2, pin_memory=True,
                                      drop_last=True)
    else:
        tr_ld = data_utils.DataLoader(tr_ds, batch_size=config.CNN_BATCH_SIZE,
                                      shuffle=True, num_workers=2, pin_memory=True,
                                      drop_last=True)
    
    vl_ld = data_utils.DataLoader(vl_ds, batch_size=config.CNN_BATCH_SIZE * 2,
                                  shuffle=False, num_workers=2, pin_memory=True)

    bs_eval = next((orig_to_eval[i] for i,n in enumerate(label_names_list)
                    if "brown" in n.lower() and "spot" in n.lower()
                    and i in orig_to_eval), 0)
    lb_eval = next((orig_to_eval[i] for i,n in enumerate(label_names_list)
                    if "leaf" in n.lower() and "blast" in n.lower()
                    and i in orig_to_eval), 2)
    
    model = EfficientNetDiseaseClassifier(
        nc, config.CNN_DROPOUT, bs_idx=bs_eval, lb_idx=lb_eval).to(dev)

    opt = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": config.CNN_LR * 0.1},
        {"params": model.head.parameters(),     "lr": config.CNN_LR},
    ], weight_decay=config.CNN_WEIGHT_DECAY)

    total_steps  = config.CNN_EPOCHS * len(tr_ld)
    warmup_steps = config.CNN_WARMUP_EPOCHS * len(tr_ld)
    def _lr_lambda(step: int) -> float:
        if step < warmup_steps: return float(step) / max(1, warmup_steps)
        prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.01, 0.5 * (1 + np.cos(np.pi * prog)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda)
    
    cls_counts = Counter(orig_to_eval[it.label] for it in tr if it.label in orig_to_eval)
    max_cnt    = max(cls_counts.values()) if cls_counts else 1
    cw         = torch.tensor([max_cnt / (cls_counts.get(i, 1) + 1e-8)
                                for i in range(nc)], dtype=torch.float32).to(dev)
    
    # UPDATED: Boost BS and LB weights
    cw[bs_eval] *= 1.3
    cw[lb_eval] *= 1.3
    
    crit_w     = FocalLoss(gamma=config.CNN_FOCAL_GAMMA, weight=cw,
                           label_smoothing=config.CNN_LABEL_SMOOTH)
    crit_bin   = FocalLoss(gamma=config.CNN_FOCAL_GAMMA + 0.5, label_smoothing=0.05)

    hist = {"epoch":[], "train_loss":[], "train_acc":[], "val_acc":[], "lr":[]}
    best_val_acc = 0.0; best_state = None
    _step = 0

    print(f"\n Training EfficientNet-B5 [{model_tag}] (Enhanced BS/LB focus)"
          f"  ({len(tr)} train | {len(vl)} val | {nc} classes | {config.CNN_EPOCHS} epochs)")
    print(f"   CutMix α={config.CNN_CUTMIX_ALPHA}  Focal γ={config.CNN_FOCAL_GAMMA}  "
          f"TwoHead w={config.CNN_TWOHEAD_WEIGHT}  BS/LB weight boost=1.3x")

    for epoch in range(config.CNN_EPOCHS):
        model.train(); rl = 0.0; correct = 0; n_seen = 0
        for xb, yb in tr_ld:
            xb, yb = xb.to(dev), yb.to(dev)
            xm, ya, yb2, lam = _cutmix_batch(xb, yb, config.CNN_CUTMIX_ALPHA)
            opt.zero_grad()
            out_main, out_aux = model(xm)
            
            loss_main = lam * crit_w(out_main, ya) + (1-lam) * crit_w(out_main, yb2)
            
            bs_lb_mask = ((ya == model.bs_idx) | (ya == model.lb_idx))
            if bs_lb_mask.any():
                ya_bin = (ya[bs_lb_mask] == model.lb_idx).long()
                yb2_bin = (yb2[bs_lb_mask] == model.lb_idx).long()
                
                crit_bin_bslb = FocalLoss(gamma=3.0, label_smoothing=0.05)
                loss_aux = (lam * crit_bin_bslb(out_aux[bs_lb_mask], ya_bin) +
                           (1-lam) * crit_bin_bslb(out_aux[bs_lb_mask], yb2_bin))
                
                loss = (0.6)*loss_main + 0.4*loss_aux
            else:
                loss = loss_main
            
            loss.backward(); opt.step(); sched.step(); _step += 1
            rl      += loss.item() * xb.size(0)
            correct += (out_main.argmax(1) == ya).sum().item()
            n_seen  += xb.size(0)

        model.eval(); v_correct = 0; v_total = 0
        with torch.no_grad():
            for xb, yb in vl_ld:
                xb, yb = xb.to(dev), yb.to(dev)
                out, _ = model(xb)
                v_correct += (out.argmax(1) == yb).sum().item()
                v_total   += yb.size(0)

        train_acc = correct / max(n_seen, 1)
        val_acc   = v_correct / max(v_total, 1)
        cur_lr    = opt.param_groups[1]["lr"]
        hist["epoch"].append(epoch+1);     hist["train_loss"].append(rl/n_seen)
        hist["train_acc"].append(train_acc); hist["val_acc"].append(val_acc)
        hist["lr"].append(cur_lr)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(f"   [{model_tag}] Ep {epoch+1:>2}/{config.CNN_EPOCHS} | "
              f"loss={rl/n_seen:.4f} tr_acc={train_acc*100:.1f}% "
              f"val_acc={val_acc*100:.1f}% lr={cur_lr:.6f}"
              + (" ★" if val_acc == best_val_acc else ""))

    if best_state is not None:
        model.load_state_dict({k: v.to(dev) for k, v in best_state.items()})
        print(f" Best val_acc={best_val_acc*100:.1f}% restored")
    model.eval()

    ep = hist["epoch"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(ep, hist["train_loss"], "o-", color="#e74c3c", lw=2, ms=5)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title(f"CNN [{model_tag}] — Training Loss")

    axes[1].plot(ep, [a*100 for a in hist["train_acc"]], "s-", color="#4f86c6",
                 lw=2, ms=5, label="Train")
    axes[1].plot(ep, [a*100 for a in hist["val_acc"]], "^-", color="#f5a623",
                 lw=2, ms=5, label="Val")
    axes[1].axhline(best_val_acc*100, color="crimson", ls="--", lw=1.2,
                    label=f"Best val={best_val_acc*100:.1f}%")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
    axes[1].set_title(f"CNN [{model_tag}] — Accuracy"); axes[1].legend()
    axes[1].set_ylim(0, 108)

    axes[2].plot(ep, hist["lr"], "^-", color="#7ed321", lw=2, ms=5)
    axes[2].axvline(config.CNN_WARMUP_EPOCHS, color="gray", ls=":", lw=1.2,
                    label=f"Warmup end (ep {config.CNN_WARMUP_EPOCHS})")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Learning Rate")
    axes[2].set_title(f"CNN [{model_tag}] — LR Schedule")
    axes[2].set_yscale("log"); axes[2].legend()

    fig.suptitle(f"EfficientNet-B5 [{model_tag}] — Enhanced BS/LB Focus  "
                 f"(best_val={best_val_acc*100:.1f}%)", fontsize=13)
    plt.tight_layout()
    _save(fig, f"03a_cnn_{model_tag}_training.png")

    hist_path = os.path.join(config.LOGS_DIR, f"cnn_{model_tag}_history.json")
    with open(hist_path, "w") as f: json.dump(hist, f, indent=2)
    print(f"{hist_path}")
    return model

@torch.no_grad()
def predict_cnn_tta(model: EfficientNetDiseaseClassifier,
                    image_pil: Image.Image,
                    num_classes: int,
                    tta_transforms: Optional[List[T.Compose]] = None) -> np.ndarray:
    model.eval()
    dev   = next(model.parameters()).device
    tfms  = tta_transforms or [_make_val_transform(config.CNN_IMG_SIZE)]
    img   = image_pil.convert("RGB")
    probs_list = []
    for tf in tfms:
        x    = tf(img).unsqueeze(0).to(dev)
        logits = model.predict(x)
        probs_list.append(torch.softmax(logits, dim=1).cpu().numpy()[0])
    return np.mean(probs_list, axis=0).astype(np.float32)

# ── Binary Arbiter: Brown Spot ↔ Leaf Blast ───────────────────────────────

class BinaryArbiterCNN(nn.Module):
    def __init__(self, dropout: float = 0.40):
        super().__init__()
        self.backbone = timm.create_model(
            "efficientnet_b2", pretrained=True, num_classes=0,
            global_pool="avg", drop_rate=dropout)
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 64),  nn.BatchNorm1d(64),  nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Linear(64, 2))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))

def train_binary_arbiter(
        train_items: List["PlantImageData"],
        val_items:   List["PlantImageData"],
        bs_label_idx: int,
        lb_label_idx: int) -> Optional[BinaryArbiterCNN]:

    dev = torch.device(device)
    bin_map = {bs_label_idx: 0, lb_label_idx: 1}

    tr = [it for it in train_items if it.label in bin_map]
    vl = [it for it in val_items   if it.label in bin_map]
    if len(tr) < 10:
        print("Arbiter: insufficient samples, skipping."); return None

    img_sz = config.CNN_IMG_SIZE
    tr_tf  = T.Compose([
        T.Resize((img_sz + 48, img_sz + 48)),
        T.RandomCrop(img_sz),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.4),
        T.RandomRotation(degrees=45),
        T.ColorJitter(brightness=0.45, contrast=0.45, saturation=0.40, hue=0.15),
        T.RandomPerspective(distortion_scale=0.25, p=0.5),
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        T.ToTensor(),
        T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        T.RandomErasing(p=0.35, scale=(0.02, 0.15)),
    ])
    vl_tf  = _make_val_transform(img_sz)

    tr_ds  = RiceImageDataset(tr, bin_map, tr_tf)
    vl_ds  = RiceImageDataset(vl, bin_map, vl_tf)
    tr_ld  = data_utils.DataLoader(tr_ds, batch_size=config.ARBITER_BATCH_SIZE,
                                   shuffle=True, num_workers=2, drop_last=False)
    vl_ld  = data_utils.DataLoader(vl_ds, batch_size=config.ARBITER_BATCH_SIZE * 2,
                                   shuffle=False, num_workers=2)

    model = BinaryArbiterCNN(config.ARBITER_DROPOUT).to(dev)
    opt   = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": config.ARBITER_LR * 0.1},
        {"params": model.head.parameters(),     "lr": config.ARBITER_LR},
    ], weight_decay=1e-4)

    total_steps  = config.ARBITER_EPOCHS * len(tr_ld)
    warmup_steps = 2 * len(tr_ld)
    def _lr_lambda(step):
        if step < warmup_steps: return float(step) / max(1, warmup_steps)
        prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.01, 0.5 * (1 + np.cos(np.pi * prog)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda)

    bs_n  = sum(1 for it in tr if it.label == bs_label_idx)
    lb_n  = sum(1 for it in tr if it.label == lb_label_idx)
    total = bs_n + lb_n
    cw    = torch.tensor([total/(2*bs_n+1e-8), total/(2*lb_n+1e-8)], device=dev)
    crit  = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)

    best_val_acc = 0.0; best_state = None; _step = 0
    hist = {"epoch":[], "train_acc":[], "val_acc":[]}

    print(f"\n⚔️  Training Binary Arbiter [Brown Spot vs Leaf Blast]  "
          f"({len(tr)} train, {len(vl)} val, {config.ARBITER_EPOCHS} epochs)")
    print(f"   BS:{bs_n}  LB:{lb_n}  class_weights=[{total/(2*bs_n+1e-8):.2f},{total/(2*lb_n+1e-8):.2f}]")

    for epoch in range(config.ARBITER_EPOCHS):
        model.train(); correct = 0; n = 0
        for xb, yb in tr_ld:
            xb, yb = xb.to(dev), yb.to(dev)
            xm, ya, yb2, lam = _cutmix_batch(xb, yb, alpha=0.25)
            opt.zero_grad()
            out  = model(xm)
            loss = lam * crit(out, ya) + (1 - lam) * crit(out, yb2)
            loss.backward(); opt.step(); sched.step(); _step += 1
            correct += (out.argmax(1) == ya).sum().item(); n += xb.size(0)

        model.eval(); vc = vt = 0
        with torch.no_grad():
            for xb, yb in vl_ld:
                xb, yb = xb.to(dev), yb.to(dev)
                vc += (model(xb).argmax(1) == yb).sum().item(); vt += yb.size(0)
        va = vc / max(vt, 1)
        hist["epoch"].append(epoch+1)
        hist["train_acc"].append(correct/max(n,1))
        hist["val_acc"].append(va)
        if va > best_val_acc:
            best_val_acc = va
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"   Arbiter Ep {epoch+1:>2}/{config.ARBITER_EPOCHS} | "
              f"tr={correct/max(n,1)*100:.1f}% val={va*100:.1f}%"
              + (" ★" if va == best_val_acc else ""))

    if best_state is not None:
        model.load_state_dict({k: v.to(dev) for k, v in best_state.items()})
    model.eval()
    print(f"Arbiter best_val={best_val_acc*100:.1f}%")

    hist_path = os.path.join(config.LOGS_DIR, "arbiter_history.json")
    with open(hist_path, "w") as f: json.dump(hist, f, indent=2)

    ep = hist["epoch"]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(ep, [a*100 for a in hist["train_acc"]], "s-", color="#4f86c6", lw=2, label="Train")
    ax.plot(ep, [a*100 for a in hist["val_acc"]],   "o-", color="#f5a623", lw=2, label="Val")
    ax.axhline(best_val_acc*100, color="crimson", ls="--", lw=1.2,
               label=f"Best={best_val_acc*100:.1f}%")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
    ax.set_title("Binary Arbiter [Brown Spot vs Leaf Blast] — Training Curves")
    ax.legend(); ax.set_ylim(0, 108)
    plt.tight_layout(); _save(fig, "03b_arbiter_training.png")
    return model

@torch.no_grad()
def predict_binary_arbiter(model: BinaryArbiterCNN,
                           image_pil: Image.Image,
                           tta_transforms: Optional[List[T.Compose]] = None) -> np.ndarray:
    model.eval()
    dev  = next(model.parameters()).device
    tfms = tta_transforms or [_make_val_transform(config.CNN_IMG_SIZE)]
    img  = image_pil.convert("RGB")
    plist = []
    for tf in tfms:
        x = tf(img).unsqueeze(0).to(dev)
        plist.append(torch.softmax(model(x), dim=1).cpu().numpy()[0])
    return np.mean(plist, axis=0).astype(np.float32)

# SECTION 13.1-B: NEURAL FUSION MODEL

class NeuralFusionModel(nn.Module):
    def __init__(self, num_classes: int, extra_features_dim: int = 3,
                 use_cnn_probs: bool = True):
        super().__init__()
        self.use_cnn_probs = use_cnn_probs
        in_dim = num_classes * (2 if use_cnn_probs else 1) + extra_features_dim
        self.drop = nn.Dropout(config.NN_FUSION_DROPOUT)
        self.fc1  = nn.Linear(in_dim, 256)
        self.bn1  = nn.BatchNorm1d(256)
        self.fc2  = nn.Linear(256, 128)
        self.bn2  = nn.BatchNorm1d(128)
        self.skip = nn.Linear(in_dim, 128)
        self.fc3  = nn.Linear(128, 64)
        self.bn3  = nn.BatchNorm1d(64)
        self.out  = nn.Linear(64, num_classes)
        self.act  = nn.GELU()
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.skip(x)
        h = self.act(self.bn1(self.fc1(x)));    h = self.drop(h)
        h = self.act(self.bn2(self.fc2(h)) + identity); h = self.drop(h)
        h = self.act(self.bn3(self.fc3(h)));    h = self.drop(h)
        return self.out(h)

def build_fusion_feature_vector(label_votes, crop_context,
                                 eval_class_indices, label_names_list, orig_to_eval,
                                 cnn_probs: Optional[np.ndarray] = None):
    if not label_votes: return None
    nc  = len(eval_class_indices); vec = np.zeros(nc, dtype=np.float32)
    for orig_idx, eval_idx in orig_to_eval.items():
        vec[eval_idx] = float(label_votes.get(label_names_list[orig_idx], 0.0))
    extra = np.array([float(crop_context.get("top_crop_score_norm",0.5)),
                      float(crop_context.get("n_crops_norm",0.5)),
                      float(crop_context.get("month_norm",0.5))], dtype=np.float32)
    if cnn_probs is not None and len(cnn_probs) == nc:
        return np.concatenate([vec, cnn_probs.astype(np.float32), extra])
    return np.concatenate([vec, extra])

def get_current_crop_context(rag2):
    now  = datetime.datetime.now()
    recs = rag2.recommend({"temperature":26.0,"humidity":75.0,"rainfall":1000.0,"ph":6.5}, top_n=3)
    ts   = recs[0].score if recs else 0.5
    return {"top_crop_score_norm": float(np.clip(ts,0,1)),
            "n_crops_norm":        float(np.clip(len(rag2.unique_crops)/20.0,0,1)),
            "month_norm":          (now.month-1)/11.0}

def train_neural_fusion_model(train_images, rag_system, rag2,
                               eval_class_indices, label_names_list, orig_to_eval,
                               cnn_model: Optional[EfficientNetDiseaseClassifier] = None):
    nc  = len(eval_class_indices); dev = torch.device(device)
    X_list, y_list = [], []
    rng = np.random.default_rng(42)
    indices = np.arange(len(train_images))
    if len(indices) > config.NN_FUSION_MAX_SAMPLES:
        indices = rng.choice(indices, size=config.NN_FUSION_MAX_SAMPLES, replace=False)
    crop_ctx = get_current_crop_context(rag2)

    cnn_tta_tfms = _make_tta_transforms(config.CNN_IMG_SIZE) if cnn_model else None

    for idx in tqdm(indices, desc="Fusion features"):
        item = train_images[idx]
        if item.label not in orig_to_eval: continue
        cases  = rag_system.find_similar_cases(item.image_pil, k=config.TOP_K_SIMILAR, is_query=False)
        rag_an = rag_system.analyze_similar_cases(cases)
        lv     = rag_an.get("label_votes",{})
        if not lv: continue
        cp = None
        if cnn_model is not None:
            try: cp = predict_cnn_tta(cnn_model, item.image_pil, nc, cnn_tta_tfms[:2])
            except: cp = None
        fv = build_fusion_feature_vector(lv, crop_ctx, eval_class_indices,
                                          label_names_list, orig_to_eval, cp)
        if fv is None: continue
        X_list.append(fv); y_list.append(orig_to_eval[item.label])
    if not X_list: print("  Fusion: no samples."); return None

    use_cnn_p  = (cnn_model is not None)
    X = torch.tensor(np.stack(X_list), dtype=torch.float32)
    y = torch.tensor(y_list, dtype=torch.long)
    ds = data_utils.TensorDataset(X, y)
    ld = data_utils.DataLoader(ds, batch_size=config.NN_FUSION_BATCH_SIZE,
                               shuffle=True, drop_last=False)
    model = NeuralFusionModel(nc, extra_features_dim=3, use_cnn_probs=use_cnn_p).to(dev)
    opt   = torch.optim.AdamW(model.parameters(), lr=config.NN_FUSION_LR,
                               weight_decay=2e-4, betas=(0.9, 0.999))
    total_steps  = config.NN_FUSION_EPOCHS * len(ld)
    warmup_steps = 2 * len(ld)
    def _lr_lambda(step):
        if step < warmup_steps: return float(step) / max(1, warmup_steps)
        prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.05, 0.5 * (1 + np.cos(np.pi * prog)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda)
    crit  = nn.CrossEntropyLoss(label_smoothing=0.10)

    hist = {"epoch":[], "loss":[], "accuracy":[], "lr":[]}
    print(f"\n Training Neural Fusion  ({len(ds)} samples | {nc} classes | "
          f"{config.NN_FUSION_EPOCHS} epochs | CNN probs={'YES' if use_cnn_p else 'NO'})")
    _step = 0
    for epoch in range(config.NN_FUSION_EPOCHS):
        model.train(); rl = 0.0; correct = 0
        for xb, yb in ld:
            xb, yb = xb.to(dev), yb.to(dev)
            opt.zero_grad()
            logits = model(xb); loss = crit(logits, yb)
            loss.backward(); opt.step(); sched.step(); _step += 1
            rl += loss.item() * xb.size(0)
            correct += (logits.argmax(1) == yb).sum().item()
        avg_loss = rl / len(ds); avg_acc = correct / len(ds)
        cur_lr   = opt.param_groups[0]["lr"]
        hist["epoch"].append(epoch+1); hist["loss"].append(avg_loss)
        hist["accuracy"].append(avg_acc); hist["lr"].append(cur_lr)
        print(f"   Fusion Ep {epoch+1:>2}/{config.NN_FUSION_EPOCHS} | "
              f"loss={avg_loss:.4f} acc={avg_acc*100:.1f}% lr={cur_lr:.6f}")
    model.eval(); print("Neural fusion trained.")

    hist_path = os.path.join(config.LOGS_DIR, "fusion_training_history.json")
    with open(hist_path, "w") as f: json.dump(hist, f, indent=2)
    print(f"  {hist_path}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    ep = hist["epoch"]
    axes[0].plot(ep, hist["loss"], "o-", color="#e74c3c", lw=2, ms=6)
    axes[0].fill_between(ep, hist["loss"], alpha=0.12, color="#e74c3c")
    min_idx = int(np.argmin(hist["loss"]))
    axes[0].annotate(f'min={hist["loss"][min_idx]:.4f}',
                     xy=(ep[min_idx], hist["loss"][min_idx]),
                     xytext=(0, 14), textcoords="offset points",
                     ha="center", fontsize=8, color="#e74c3c",
                     arrowprops=dict(arrowstyle="->", color="#e74c3c", lw=1.2))
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("CE Loss")
    axes[0].set_title("Fusion — Training Loss")
    acc_pct = [a*100 for a in hist["accuracy"]]
    axes[1].plot(ep, acc_pct, "s-", color="#4f86c6", lw=2, ms=6)
    axes[1].fill_between(ep, acc_pct, alpha=0.12, color="#4f86c6")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
    axes[1].set_title("Fusion — Training Accuracy"); axes[1].set_ylim(0, 108)
    axes[2].plot(ep, hist["lr"], "^-", color="#7ed321", lw=2, ms=6)
    axes[2].axvline(2, color="crimson", ls="--", lw=1.2, label="Warmup end")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Learning Rate")
    axes[2].set_title("Fusion — LR Schedule"); axes[2].set_yscale("log")
    axes[2].legend(fontsize=8)
    fig.suptitle("Neural Fusion — RAG+CNN Combined Inputs", fontsize=13)
    plt.tight_layout(); _save(fig, "03c_fusion_training_curves.png")
    return model

# SECTION 13.2: AGENT 1 — DISEASE PREDICTION

class DiseasePredictionAgent:
    AGENT_NAME = "RiceDiseasePredictionAgent"
    def __init__(self, rag_system, processor):
        self.rag = rag_system; self.processor = processor

    def _calibrated_confidence(self, cases, rag_analysis, best_weight):
        if not cases: return 0.0
        top_sim = cases[0][1]; avg_sim = rag_analysis.get("avg_similarity",0.0)
        votes = rag_analysis.get("label_votes",{}); vals = sorted(votes.values(),reverse=True)
        gap   = (vals[0]-vals[1]) if len(vals)>1 else (vals[0] if vals else 0)
        raw   = (0.30*top_sim + 0.25*avg_sim + 0.25*gap
                 + 0.10*min(len(cases)/config.TOP_K_SIMILAR,1.0)
                 + 0.10*float(np.exp(-np.var([s for _,s in cases])*8)))
        return float(np.clip(1.0/(1.0+np.exp(-12.0*(raw-0.45))), 0.0, 1.0))

    def run(self, image_pil, rag_analysis=None, is_query=True):
        sc = self.rag.find_similar_cases(image_pil, k=config.TOP_K_SIMILAR, is_query=is_query)
        if rag_analysis is None: rag_analysis = self.rag.analyze_similar_cases(sc)
        lv = rag_analysis.get("label_votes",{})
        if not lv: return DiagnosisResult("Unknown","Rice",0.0,False,-1,["No cases."],self.AGENT_NAME)
        best_lbl, best_w = max(lv.items(), key=lambda x: x[1])
        meta      = self.processor.mapping.get(best_lbl, {})
        pred_lbl  = (self.processor.label_names.index(best_lbl)
                     if best_lbl in self.processor.label_names else -1)
        conf      = self._calibrated_confidence(sc, rag_analysis, best_w)
        top_sim   = sc[0][1] if sc else 0.0
        evid = [f"RAG1 best='{best_lbl}' consensus={best_w:.3f}",
                f"RAG1 top_sim={top_sim:.3f} avg_sim={rag_analysis.get('avg_similarity',0):.3f}",
                f"RAG1 {len(sc)}/{config.TOP_K_SIMILAR} cases"]
        vlm_r = None
        if config.VLM_ENABLED:
            if not vlm_reranker.loaded: vlm_reranker.load()
            if vlm_reranker.loaded:
                try: vlm_r = vlm_reranker.rerank(image_pil, dict(lv), self.processor.mapping, self.processor.label_names)
                except Exception as e: logger.warning(f"VLM failed: {e}")
        if vlm_r is not None:
            return DiagnosisResult(vlm_r.disease,"Rice",vlm_r.confidence,vlm_r.is_healthy,
                                   vlm_r.predicted_label, evid+vlm_r.evidence,
                                   f"{self.AGENT_NAME}+VLM(BLIP-2)")
        return DiagnosisResult(meta.get("disease",best_lbl),"Rice",conf,
                               meta.get("healthy",False),pred_lbl,evid,self.AGENT_NAME)

# SECTION 14: AGENT 2 — CROP RECOMMENDATION AGENT


class CropRecommendationAgent:
    AGENT_NAME = "CropRecommendationAgent"
    def __init__(self, rag2): self.rag2 = rag2

    def run(self, location=config.DEFAULT_LOCATION, latitude=config.DEFAULT_LAT,
            longitude=config.DEFAULT_LON, temperature=None, humidity=None,
            rainfall=None, ph=None, nitrogen=None, phosphorus=None,
            potassium=None, soil_type=None, season=None, top_n=None):
        query: Dict[str,Any] = {"region": location}
        for k, v in [("latitude",latitude),("longitude",longitude),
                     ("temperature",temperature),("humidity",humidity),
                     ("rainfall",rainfall),("ph",ph),("nitrogen",nitrogen),
                     ("phosphorus",phosphorus),("potassium",potassium),
                     ("soil_type",soil_type),("season",season)]:
            if v is not None: query[k] = v
        recs = self.rag2.recommend(query, top_n=top_n or config.TOP_N_CROPS)
        return {"agent":self.AGENT_NAME,"location":location,"query_conditions":query,
                "rag_source":"RAG2 — SmartFarming CSV + RF + FAISS",
                "top_recommendations":[{"rank":i+1,"crop":r.crop,"score":r.score,
                    "confidence_pct":r.confidence_pct,"season":r.season,
                    "best_soil":r.soil_match,"climate_match":r.climate_match,
                    "reasoning":r.reasoning} for i,r in enumerate(recs)]}

#SECTION 15: AGENTIC ORCHESTRATOR

disease_agent = DiseasePredictionAgent(rag_system, processor)
crop_agent    = CropRecommendationAgent(rag2=rag2_crop_recommender)

class AgenticOrchestrator:
    def __init__(self, rag, disease_agent, crop_agent,
                 fusion_model=None,
                 cnn_model: Optional[EfficientNetDiseaseClassifier] = None,
                 arbiter_model: Optional[BinaryArbiterCNN] = None,
                 eval_class_indices: Optional[List[int]] = None,
                 bs_eval_idx: int = 0,
                 lb_eval_idx: int = 2):
        self.rag           = rag
        self.disease_agent = disease_agent
        self.crop_agent    = crop_agent
        self.fusion_model  = fusion_model
        self.cnn_model     = cnn_model
        self.arbiter_model = arbiter_model
        self.eval_class_indices = eval_class_indices or []
        self.bs_eval_idx   = bs_eval_idx
        self.lb_eval_idx   = lb_eval_idx
        self._tta_tfms     = _make_tta_transforms(config.CNN_IMG_SIZE)
        self._nc           = len(self.eval_class_indices)

    def _cnn_probs(self, image_pil: Image.Image) -> Optional[np.ndarray]:
        if self.cnn_model is None or not config.USE_CNN_CLASSIFIER: return None
        try:
            return predict_cnn_tta(self.cnn_model, image_pil,
                                   self._nc, self._tta_tfms)
        except Exception as e:
            logger.warning(f"CNN inference failed: {e}"); return None

    def _apply_arbiter(self, blended_probs: np.ndarray,
                       image_pil: Image.Image) -> np.ndarray:
        if (self.arbiter_model is None or not config.USE_BINARY_ARBITER
                or len(blended_probs) <= max(self.bs_eval_idx, self.lb_eval_idx)):
            return blended_probs
        p_bs = blended_probs[self.bs_eval_idx]
        p_lb = blended_probs[self.lb_eval_idx]
        sorted_probs = np.argsort(blended_probs)[::-1]
        top2 = set(sorted_probs[:2].tolist())
        if ({self.bs_eval_idx, self.lb_eval_idx} != top2):
            return blended_probs
        if abs(p_bs - p_lb) >= config.ARBITER_AMBIGUITY_THRESH:
            return blended_probs
        try:
            arb_p = predict_binary_arbiter(self.arbiter_model, image_pil,
                                           self._tta_tfms[:3])
            refined = blended_probs.copy()
            w = config.ARBITER_WEIGHT
            refined[self.bs_eval_idx] = (1-w)*p_bs + w*arb_p[0]
            refined[self.lb_eval_idx] = (1-w)*p_lb + w*arb_p[1]
            refined = np.clip(refined, 0, None)
            refined /= (refined.sum() + 1e-8)
            return refined
        except Exception as e:
            logger.warning(f"Arbiter failed: {e}"); return blended_probs

    def _apply_neural_fusion(self, diagnosis, rag_analysis,
                              cnn_probs: Optional[np.ndarray] = None):
        if self.fusion_model is None: return diagnosis
        lv = rag_analysis.get("label_votes",{})
        if not lv: return diagnosis
        ctx = get_current_crop_context(rag2_crop_recommender)
        fv  = build_fusion_feature_vector(lv, ctx, EVAL_CLASS_INDICES, label_names,
                                           ORIG_TO_EVAL, cnn_probs)
        if fv is None: return diagnosis
        x = torch.tensor(fv[None,:], dtype=torch.float32, device=device)
        with torch.no_grad():
            probs = torch.softmax(self.fusion_model(x), dim=1).cpu().numpy()[0]
        probs = self._apply_arbiter(probs, diagnosis.evidence[0]
                                    if False else None) if False else probs
        bei  = int(np.argmax(probs))
        borig = EVAL_CLASS_INDICES[bei]; blbl = label_names[borig]
        meta = processor.mapping.get(blbl, {})
        return DiagnosisResult(
            disease=meta.get("disease", blbl), plant_species="Rice",
            confidence=float(probs[bei]),
            is_healthy=meta.get("healthy", False), predicted_label=borig,
            evidence=list(diagnosis.evidence) +
                     [f"NeuralFusion→'{blbl}' p={float(probs[bei]):.3f}"],
            agent_name=diagnosis.agent_name + "+NeuralFusion")

    def diagnose(self, image_pil, use_rag=True,
                 location=config.DEFAULT_LOCATION, latitude=config.DEFAULT_LAT,
                 longitude=config.DEFAULT_LON, temperature=None, humidity=None,
                 rainfall=None, ph=None, soil_type=None, season=None):
        cnn_probs = self._cnn_probs(image_pil)

        ra: Dict = {}
        if use_rag:
            cases = self.rag.find_similar_cases(image_pil, is_query=True)
            ra    = self.rag.analyze_similar_cases(cases)
        diag = self.disease_agent.run(image_pil, ra, is_query=True)
        crop = self.crop_agent.run(location=location, latitude=latitude,
                                   longitude=longitude, temperature=temperature,
                                   humidity=humidity, rainfall=rainfall,
                                   ph=ph, soil_type=soil_type, season=season)

        if cnn_probs is not None and ra:
            rag_votes   = ra.get("label_votes", {})
            rag_vec     = np.zeros(self._nc, dtype=np.float32)
            for orig_idx, ei in ORIG_TO_EVAL.items():
                rag_vec[ei] = float(rag_votes.get(label_names[orig_idx], 0.0))
            rag_norm = rag_vec / (rag_vec.sum() + 1e-8)
            blended  = (config.CNN_ENSEMBLE_WEIGHT * cnn_probs
                        + config.RAG_ENSEMBLE_WEIGHT * rag_norm)
            blended  = np.clip(blended, 0, None)
            blended /= (blended.sum() + 1e-8)

            blended = self._apply_arbiter(blended, image_pil)

            bei   = int(np.argmax(blended))
            borig = EVAL_CLASS_INDICES[bei]; blbl  = label_names[borig]
            meta  = processor.mapping.get(blbl, {})
            cnn_diag = DiagnosisResult(
                disease=meta.get("disease", blbl), plant_species="Rice",
                confidence=float(blended[bei]),
                is_healthy=meta.get("healthy", False), predicted_label=borig,
                evidence=(list(diag.evidence) +
                          [f"CNN_prob={cnn_probs[bei]:.3f} RAG_prob={rag_norm[bei]:.3f}",
                           f"Blended→'{blbl}' p={blended[bei]:.3f}"]),
                agent_name=diag.agent_name + "+CNN_B5+Arbiter")
            diag = cnn_diag
        elif cnn_probs is not None:
            bei   = int(np.argmax(cnn_probs))
            borig = EVAL_CLASS_INDICES[bei]; blbl  = label_names[borig]
            meta  = processor.mapping.get(blbl, {})
            diag  = DiagnosisResult(
                disease=meta.get("disease", blbl), plant_species="Rice",
                confidence=float(cnn_probs[bei]),
                is_healthy=meta.get("healthy", False), predicted_label=borig,
                evidence=[f"CNN_B5_only→'{blbl}' p={cnn_probs[bei]:.3f}"],
                agent_name="CNN_B5")

        try:
            if self.fusion_model is not None and ra:
                diag = self._apply_neural_fusion(diag, ra, cnn_probs)
        except Exception as e:
            logger.warning(f"Fusion failed: {e}")

        rag_src = "CNN_B5+Arbiter+RAG1+BLIP2+NeuralFusion" if (
            cnn_probs is not None and config.VLM_ENABLED and self.fusion_model) else (
            "CNN_B5+Arbiter+RAG1+NeuralFusion" if (
                cnn_probs is not None and self.fusion_model) else (
            "RAG1+BLIP2+NeuralFusion" if (
                config.VLM_ENABLED and self.fusion_model) else "RAG1"))

        return {"final_diagnosis": {"disease":       diag.disease,
                                    "plant_species": "Rice",
                                    "is_healthy":    diag.is_healthy,
                                    "confidence":    diag.confidence,
                                    "predicted_label": diag.predicted_label},
                "disease_agent": {"name":       diag.agent_name,
                                  "result":     {"disease":    diag.disease,
                                                 "confidence": diag.confidence},
                                  "evidence":   diag.evidence,
                                  "rag_source": rag_src},
                "crop_agent":    {"name":       crop_agent.AGENT_NAME,
                                  "result":     crop,
                                  "rag_source": "RAG2"},
                "rag_analysis":  ra}

# SECTION 15.1: TRAIN ALL MODELS
train_images = index_images
val_images   = pca_fit_images
print(f"\n   CNN train set : {len(train_images):,} images")
print(f"   CNN val   set : {len(val_images):,} images")

cnn_model = None
if config.USE_CNN_CLASSIFIER:
    print("\n" + "="*60)
    print(" TRAINING EfficientNet-B5 (4-class disease classifier)")
    print("="*60)
    cnn_model = train_efficientnet_classifier(
        train_items=train_images,
        val_items=val_images,
        eval_class_indices=EVAL_CLASS_INDICES,
        label_names_list=label_names,
        orig_to_eval=ORIG_TO_EVAL,
        model_tag="main")
else:
    print(" CNN classifier disabled.")

BS_ORIG_IDX = next((i for i,n in enumerate(label_names)
                    if "brown" in n.lower() and "spot" in n.lower()), None)
LB_ORIG_IDX = next((i for i,n in enumerate(label_names)
                    if "leaf" in n.lower() and "blast" in n.lower()), None)
BS_EVAL_IDX = ORIG_TO_EVAL.get(BS_ORIG_IDX, 0) if BS_ORIG_IDX is not None else 0
LB_EVAL_IDX = ORIG_TO_EVAL.get(LB_ORIG_IDX, 2) if LB_ORIG_IDX is not None else 2
print(f"\n Brown Spot: orig={BS_ORIG_IDX} eval={BS_EVAL_IDX}  "
      f"| Leaf Blast: orig={LB_ORIG_IDX} eval={LB_EVAL_IDX}")

arbiter_model = None
if config.USE_BINARY_ARBITER and BS_ORIG_IDX is not None and LB_ORIG_IDX is not None:
    print("\n" + "="*60)
    print("  TRAINING Binary Arbiter [Brown Spot vs Leaf Blast]")
    print("="*60)
    arbiter_model = train_binary_arbiter(
        train_items=train_images,
        val_items=val_images,
        bs_label_idx=BS_ORIG_IDX,
        lb_label_idx=LB_ORIG_IDX)
else:
    print(" Binary arbiter disabled or labels not found.")

fusion_model = None
if config.USE_NEURAL_FUSION:
    print("\n" + "="*60)
    print(" TRAINING Neural Fusion")
    print("="*60)
    train_eval = [it for it in train_images if it.label in ORIG_TO_EVAL]
    if not train_eval:
        print(" No train images in eval classes, skipping fusion.")
    else:
        fusion_model = train_neural_fusion_model(
            train_images=train_eval, rag_system=rag_system,
            rag2=rag2_crop_recommender,
            eval_class_indices=EVAL_CLASS_INDICES,
            label_names_list=label_names,
            orig_to_eval=ORIG_TO_EVAL,
            cnn_model=cnn_model)
else:
    print(" Neural fusion disabled.")

orchestrator = AgenticOrchestrator(
    rag=rag_system,
    disease_agent=disease_agent,
    crop_agent=crop_agent,
    fusion_model=fusion_model,
    cnn_model=cnn_model,
    arbiter_model=arbiter_model,
    eval_class_indices=EVAL_CLASS_INDICES,
    bs_eval_idx=BS_EVAL_IDX,
    lb_eval_idx=LB_EVAL_IDX)

#  SECTION 16: EVALUATION + ALL PLOTS

class ModelEvaluator:
    def __init__(self, orchestrator, processor, label_names,
                 eval_class_indices, eval_label_names):
        self.orchestrator       = orchestrator
        self.processor          = processor
        self.label_names        = label_names
        self.eval_class_indices = eval_class_indices
        self.eval_label_names   = eval_label_names
        self.num_eval_classes   = len(eval_class_indices)
        self.orig_to_eval       = {orig:ei for ei,orig in enumerate(eval_class_indices)}
        self.short              = [n.replace("Rice___","").replace("_"," ")
                                   for n in eval_label_names]

    def evaluate_on_test_set(self, test_data) -> Dict[str,Any]:
        print("\n"+"="*60)
        print(f" EVALUATING — {self.num_eval_classes} CLASSES")
        print("="*60)
        eval_data = [it for it in test_data if it.label in self.orig_to_eval]
        print(f"   Test samples: {len(eval_data)}")

        y_true=[]; y_pred=[]; y_scores=[]; preds=[]; confs=[]
        tracker = EmissionsTracker(
            project_name=f"RiceRAG_{self.num_eval_classes}cls",
            output_dir=config.OUTPUT_DIR, output_file="emissions.csv",
            allow_multiple_runs=True)
        tracker.start()

        for item in tqdm(eval_data, desc="Evaluating"):
            res       = self.orchestrator.diagnose(item.image_pil, use_rag=True)
            t_orig    = item.label
            p_orig    = res["final_diagnosis"]["predicted_label"]
            conf      = res["final_diagnosis"]["confidence"]
            t_eval    = self.orig_to_eval[t_orig]
            p_eval    = (self.orig_to_eval[p_orig]
                         if p_orig != -1 and p_orig in self.orig_to_eval
                         else (0 if t_eval != 0 else 1))
            prob = np.zeros(self.num_eval_classes)
            if p_orig != -1 and p_orig in self.orig_to_eval: prob[p_eval] = conf
            y_true.append(t_eval); y_pred.append(p_eval)
            y_scores.append(prob); confs.append(conf)
            preds.append({"image_id":t_orig,"true_label":item.label_name,
                          "pred_label":(self.label_names[p_orig] if p_orig!=-1 else "Unknown"),
                          "confidence":conf,"correct":t_eval==p_eval})

        emissions = tracker.stop()
        y_true = np.array(y_true); y_pred = np.array(y_pred); y_scores = np.array(y_scores)
        labs   = list(range(self.num_eval_classes))

        acc  = accuracy_score(y_true, y_pred)
        f1m  = f1_score(y_true, y_pred, average="macro",   zero_division=0, labels=labs)
        f1w  = f1_score(y_true, y_pred, average="weighted",zero_division=0, labels=labs)
        pm   = precision_score(y_true, y_pred, average="macro",   zero_division=0, labels=labs)
        pw   = precision_score(y_true, y_pred, average="weighted",zero_division=0, labels=labs)
        prec_p, rec_p, f1_p, sup_p = precision_recall_fscore_support(
            y_true, y_pred, labels=labs, zero_division=0)
        cm   = confusion_matrix(y_true, y_pred, labels=labs)
        rep  = classification_report(y_true, y_pred, labels=labs,
                                     target_names=self.eval_label_names,
                                     output_dict=True, zero_division=0)
        y_bin = label_binarize(y_true, classes=labs)
        try:    roc_auc = roc_auc_score(y_bin, y_scores, average="macro", multi_class="ovr")
        except: roc_auc = 0.0

        per_cls = {self.eval_label_names[i]:{"precision":float(prec_p[i]),"recall":float(rec_p[i]),
                                              "f1_score":float(f1_p[i]),"support":int(sup_p[i])}
                   for i in range(self.num_eval_classes)}

        metrics = {"accuracy":acc,"f1_score_macro":f1m,"f1_score_weighted":f1w,
                   "precision_macro":pm,"precision_weighted":pw,"roc_auc_macro":roc_auc,
                   "per_class_metrics":per_cls,"classification_report":rep,
                   "confusion_matrix":cm,"predictions":preds,"y_true":y_true,
                   "y_pred":y_pred,"y_scores":y_scores,"y_true_binarized":y_bin,
                   "co2_emissions_kg":emissions or 0.0,"num_eval_samples":len(eval_data),
                   "confidence_list":confs}

        print(f"\n Results: Acc={acc:.4f} | F1={f1m:.4f} | AUC={roc_auc:.4f}")
        return metrics

    def plot_confusion_matrix(self, cm) -> None:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=self.short, yticklabels=self.short,
                    linewidths=0.5, ax=axes[0], cbar_kws={"label":"Count","shrink":0.8})
        axes[0].set_title("Confusion Matrix (Counts)")
        axes[0].set_ylabel("True"); axes[0].set_xlabel("Predicted")
        axes[0].tick_params(axis="x", rotation=30)
        
        cm_n = cm.astype("float") / (cm.sum(axis=1)[:,None] + 1e-8)
        sns.heatmap(np.nan_to_num(cm_n), annot=True, fmt=".2f", cmap="YlOrRd",
                    xticklabels=self.short, yticklabels=self.short,
                    vmin=0, vmax=1, linewidths=0.5, ax=axes[1],
                    cbar_kws={"label":"Normalised","shrink":0.8})
        axes[1].set_title("Confusion Matrix (Normalised)")
        axes[1].set_ylabel("True"); axes[1].set_xlabel("Predicted")
        axes[1].tick_params(axis="x", rotation=30)
        fig.suptitle(f"Confusion Matrix — {self.num_eval_classes} Classes", fontsize=14)
        plt.tight_layout(); _save(fig, "04_confusion_matrix.png")

    def plot_metrics_dashboard(self, metrics) -> None:
        fig, axes = plt.subplots(2, 2, figsize=(18, 12))
        cls = self.eval_label_names
        cm_n = (metrics["confusion_matrix"].astype("float") /
                (metrics["confusion_matrix"].sum(axis=1)[:,None] + 1e-8))
        acc_diag = np.diag(cm_n)

        def _hbar(ax, vals, cmap, title, xlabel, vline=None):
            order = np.argsort(vals)[::-1]; lbls=[self.short[i] for i in order]; vs=[vals[i] for i in order]
            clrs = cmap(np.linspace(0.25, 0.85, len(lbls)))
            bars = ax.barh(range(len(lbls)), vs, color=clrs, edgecolor="white")
            for bar, v in zip(bars, vs):
                ax.text(bar.get_width()+0.012, bar.get_y()+bar.get_height()/2,
                        f"{v:.3f}", va="center", fontsize=9, fontweight="bold")
            ax.set_yticks(range(len(lbls))); ax.set_yticklabels(lbls)
            ax.set_xlabel(xlabel); ax.set_title(title); ax.set_xlim(0,1.15)
            ax.grid(axis="x", alpha=0.3)
            if vline is not None:
                ax.axvline(vline, color="crimson", ls="--", lw=1.5,
                           label=f"Mean={vline:.3f}"); ax.legend(fontsize=8)

        _hbar(axes[0,0], [metrics["per_class_metrics"][c]["f1_score"] for c in cls],
              plt.cm.RdYlGn, "Per-Class F1-Score", "F1")
        _hbar(axes[0,1], [metrics["per_class_metrics"][c]["precision"] for c in cls],
              plt.cm.Blues,  "Per-Class Precision", "Precision")
        _hbar(axes[1,0], [metrics["per_class_metrics"][c]["recall"] for c in cls],
              plt.cm.Purples,"Per-Class Recall", "Recall")
        _hbar(axes[1,1], acc_diag.tolist(),
              plt.cm.Oranges,"Per-Class Accuracy (CM diagonal)", "Accuracy",
              vline=float(np.mean(acc_diag)))

        fig.suptitle(f"Metrics Dashboard | Acc={metrics['accuracy']:.3f} "
                     f"F1={metrics['f1_score_macro']:.3f} AUC={metrics['roc_auc_macro']:.3f}",
                     fontsize=13)
        plt.tight_layout(); _save(fig, "05_metrics_dashboard.png")

    def plot_roc_curves(self, y_bin, y_scores) -> None:
        n = self.num_eval_classes; fpr_d={};  tpr_d={}; auc_d={}
        for i in range(n):
            try:
                fpr_d[i], tpr_d[i], _ = roc_curve(y_bin[:,i], y_scores[:,i])
                auc_d[i] = auc(fpr_d[i], tpr_d[i])
            except: fpr_d[i], tpr_d[i], auc_d[i] = [0,1],[0,1],0.0

        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        clrs = plt.cm.tab10(np.linspace(0,1,n))
        for i in range(n):
            axes[0].plot(fpr_d[i], tpr_d[i], color=clrs[i], lw=2,
                         label=f"{self.short[i]} (AUC={auc_d[i]:.3f})")
        axes[0].plot([0,1],[0,1],"k--",lw=1.5,label="Random")
        all_fpr  = np.unique(np.concatenate([fpr_d[i] for i in range(n)]))
        mean_tpr = sum(np.interp(all_fpr, fpr_d[i], tpr_d[i]) for i in range(n)) / n
        macro_a  = auc(all_fpr, mean_tpr)
        axes[0].plot(all_fpr, mean_tpr, "deeppink", lw=3, ls=":",
                     label=f"Macro-avg (AUC={macro_a:.3f})")
        axes[0].set(xlim=[0,1],ylim=[0,1.05],xlabel="FPR",ylabel="TPR",
                    title="ROC Curves — All Classes")
        axes[0].legend(loc="lower right"); axes[0].grid(alpha=0.3)

        aucs = [auc_d[i] for i in range(n)]
        bars = axes[1].bar(self.short, aucs, color=[clrs[i] for i in range(n)], edgecolor="white")
        for bar, v in zip(bars, aucs):
            axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                         f"{v:.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
        axes[1].axhline(macro_a, color="deeppink", ls="--", lw=2,
                        label=f"Macro AUC={macro_a:.3f}")
        axes[1].set_ylim(0,1.15); axes[1].set_ylabel("AUC")
        axes[1].set_title("Per-Class AUC Comparison"); axes[1].legend()
        axes[1].tick_params(axis="x", rotation=20); axes[1].grid(axis="y",alpha=0.3)

        fig.suptitle("ROC-AUC Analysis", fontsize=14)
        plt.tight_layout(); _save(fig, "06_roc_curves.png")

    def plot_per_class_prf(self, metrics) -> None:
        cls = self.eval_label_names
        prec = [metrics["per_class_metrics"][c]["precision"] for c in cls]
        rec  = [metrics["per_class_metrics"][c]["recall"]    for c in cls]
        f1   = [metrics["per_class_metrics"][c]["f1_score"]  for c in cls]
        sup  = [metrics["per_class_metrics"][c]["support"]   for c in cls]
        x = np.arange(len(self.short)); w = 0.26
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        axes[0].bar(x-w, prec, w, label="Precision", color="#4f86c6", edgecolor="white")
        axes[0].bar(x,   rec,  w, label="Recall",    color="#f5a623", edgecolor="white")
        axes[0].bar(x+w, f1,   w, label="F1",        color="#7ed321", edgecolor="white")
        axes[0].set_xticks(x); axes[0].set_xticklabels(self.short, rotation=20, ha="right")
        axes[0].set_ylim(0,1.15); axes[0].set_title("Precision / Recall / F1 by Class")
        axes[0].legend()
        for xi, (p,r,f) in enumerate(zip(prec,rec,f1)):
            for off, v in [(-w,p),(0,r),(w,f)]:
                axes[0].text(xi+off, v+0.02, f"{v:.2f}", ha="center", fontsize=7.5, fontweight="bold")

        PAL4 = ["#4f86c6","#f5a623","#7ed321","#e74c3c"]
        bars = axes[1].bar(self.short, sup, color=PAL4[:len(self.short)], edgecolor="white")
        for bar, v in zip(bars, sup):
            axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                         str(v), ha="center", va="bottom", fontsize=10, fontweight="bold")
        axes[1].set_ylabel("Support"); axes[1].set_title("Test Samples per Class")
        axes[1].tick_params(axis="x", rotation=20)

        fig.suptitle("Per-Class Classification Metrics", fontsize=14)
        plt.tight_layout(); _save(fig, "07_per_class_metrics.png")

    def plot_confidence_analysis(self, metrics) -> None:
        preds = metrics["predictions"]
        cc = [p["confidence"] for p in preds if p["correct"]]
        cw = [p["confidence"] for p in preds if not p["correct"]]
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        axes[0].hist(cc, bins=20, alpha=0.7, color="#4CAF50", edgecolor="white",
                     label=f"Correct (n={len(cc)})")
        axes[0].hist(cw, bins=20, alpha=0.7, color="#f44336", edgecolor="white",
                     label=f"Wrong (n={len(cw)})")
        axes[0].set_xlabel("Confidence"); axes[0].set_ylabel("Count")
        axes[0].set_title("Confidence: Correct vs Wrong"); axes[0].legend()

        cls_c: Dict[str, List] = {s:[] for s in self.short}
        for p in preds:
            s = p["true_label"].replace("Rice___","").replace("_"," ")
            if s in cls_c: cls_c[s].append(p["confidence"])
        bp_data = [cls_c.get(s,[0]) for s in self.short]
        bp = axes[1].boxplot(bp_data, patch_artist=True,
                             medianprops=dict(color="black",lw=2))
        PAL4 = ["#4f86c6","#f5a623","#7ed321","#e74c3c"]
        for patch, color in zip(bp["boxes"], PAL4[:len(self.short)]):
            patch.set_facecolor(color); patch.set_alpha(0.7)
        axes[1].set_xticklabels(self.short, rotation=20, ha="right")
        axes[1].set_ylabel("Confidence"); axes[1].set_title("Per-Class Confidence Box Plot")

        all_c = [p["confidence"] for p in preds]
        all_r = [1 if p["correct"] else 0 for p in preds]
        axes[2].scatter(all_c, all_r, alpha=0.25, s=15,
                        c=["#4CAF50" if r else "#f44336" for r in all_r])
        axes[2].axhline(0.5, color="gray", ls="--", lw=1)
        axes[2].set_xlabel("Confidence"); axes[2].set_ylabel("Correct (1) / Wrong (0)")
        axes[2].set_yticks([0,1]); axes[2].set_title("Confidence vs Correctness")

        fig.suptitle("Confidence Score Analysis", fontsize=14)
        plt.tight_layout(); _save(fig, "08_confidence_analysis.png")

    def plot_co2_emissions(self, emissions_kg) -> None:
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        axes[0].bar(["This Model"], [emissions_kg], color="#e74c3c",
                    edgecolor="white", lw=2, alpha=0.85, width=0.4)
        axes[0].set_ylabel("CO₂ (kg)"); axes[0].set_ylim(0, emissions_kg*2 or 1)
        axes[0].set_title(f"Inference CO₂: {emissions_kg:.6f} kg")
        axes[0].text(0, emissions_kg*1.05 if emissions_kg>0 else 0.5,
                     f"{emissions_kg:.6f} kg",
                     ha="center", fontsize=12, fontweight="bold", color="#e74c3c")
        comps = {"This\nModel":emissions_kg, "Smartphone\n(1h)":0.007,
                 "LED Bulb\n(1h)":0.008, "Car\n(1 km)":0.120, "Air\nTravel (1km)":0.255}
        si = sorted(comps.items(), key=lambda x: x[1])
        lbs, vs = zip(*si)
        clrs = ["#e74c3c" if "Model" in l else "#3498db" for l in lbs]
        bars = axes[1].barh(lbs, vs, color=clrs, edgecolor="white", alpha=0.85)
        for bar, v in zip(bars, vs):
            axes[1].text(v+max(vs)*0.02, bar.get_y()+bar.get_height()/2,
                         f"{v:.4f}" if v<0.01 else f"{v:.3f}",
                         va="center", fontsize=9, fontweight="bold")
        axes[1].set_xlabel("CO₂ (kg)"); axes[1].set_title("Emissions Comparison")
        fig.suptitle("CO₂ Emissions Analysis", fontsize=14)
        plt.tight_layout(); _save(fig, "09_co2_emissions.png")

    def plot_scorecard(self, metrics) -> None:
        fig = plt.figure(figsize=(14, 8))
        fig.patch.set_facecolor("#1a1a2e")
        gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.55, wspace=0.4)
        tiles = [("Accuracy",       f"{metrics['accuracy']:.4f}",            "#4CAF50"),
                 ("F1 Macro",       f"{metrics['f1_score_macro']:.4f}",       "#2196F3"),
                 ("F1 Weighted",    f"{metrics['f1_score_weighted']:.4f}",    "#9C27B0"),
                 ("ROC-AUC",        f"{metrics['roc_auc_macro']:.4f}",        "#FF9800"),
                 ("Prec Macro",     f"{metrics['precision_macro']:.4f}",      "#00BCD4"),
                 ("Prec Weighted",  f"{metrics['precision_weighted']:.4f}",   "#F44336"),
                 ("CO₂ (kg)",       f"{metrics['co2_emissions_kg']:.6f}",     "#8BC34A"),
                 ("Test Samples",   str(metrics['num_eval_samples']),          "#607D8B")]
        for idx, (label, value, color) in enumerate(tiles):
            row, col = divmod(idx, 4); ax = fig.add_subplot(gs[row, col])
            ax.set_facecolor("#16213e"); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
            ax.text(0.5, 0.65, value, ha="center", va="center",
                    fontsize=22, fontweight="bold", color=color, transform=ax.transAxes)
            ax.text(0.5, 0.22, label, ha="center", va="center",
                    fontsize=10, color="white", transform=ax.transAxes)
            for sp in ["top","bottom","left","right"]:
                ax.spines[sp].set_edgecolor(color); ax.spines[sp].set_linewidth(2)
                ax.spines[sp].set_visible(True)
        fig.suptitle("Rice Agentic-RAG — Evaluation Scorecard [Enhanced BS/LB Focus]",
                     fontsize=16, fontweight="bold", color="white", y=0.98)
        _save(fig, "10_scorecard.png")

    def generate_all_plots(self, metrics, location=config.DEFAULT_LOCATION) -> None:
        print(f"\n Generating all plots → {config.PLOTS_DIR}")
        recs = rag2_crop_recommender.recommend(
            {"temperature":26.0,"humidity":78.0,"rainfall":1200.0,"ph":6.2,"season":"kharif"})

        self.plot_confusion_matrix(metrics["confusion_matrix"])
        self.plot_metrics_dashboard(metrics)
        self.plot_roc_curves(metrics["y_true_binarized"], metrics["y_scores"])
        self.plot_per_class_prf(metrics)
        self.plot_confidence_analysis(metrics)
        self.plot_co2_emissions(metrics["co2_emissions_kg"])
        self.plot_scorecard(metrics)
        rag2_crop_recommender.plot_recommendations(recs, location=location)
        rag2_crop_recommender.plot_crop_condition_heatmap()

        for loc_key, q in [
            ("kharif_WB",   {"temperature":28.0,"humidity":82.0,"rainfall":1600.0,"ph":6.0,"soil_type":"alluvial","season":"kharif"}),
            ("rabi_Punjab", {"temperature":14.0,"humidity":55.0,"rainfall":350.0, "ph":7.5,"soil_type":"loam",    "season":"rabi"}),
            ("kharif_KL",   {"temperature":30.0,"humidity":90.0,"rainfall":3000.0,"ph":5.5,"soil_type":"laterite","season":"kharif"}),
        ]:
            rag2_crop_recommender.plot_recommendations(
                rag2_crop_recommender.recommend(q),
                location=loc_key,
                filename=f"13_crop_recs_{loc_key}.png")

        print(f" All plots saved to {config.PLOTS_DIR}")

    def save_metrics_report(self, metrics) -> str:
        path = os.path.join(config.OUTPUT_DIR, "evaluation_report.txt")
        recs = rag2_crop_recommender.recommend(
            {"temperature":26.0,"humidity":78.0,"rainfall":1200.0,"ph":6.2,"season":"kharif"})
        with open(path, "w", encoding="utf-8") as f:
            f.write("="*80+"\n  RICE AGENTIC-RAG — EVALUATION REPORT  (Enhanced BS/LB Focus)\n"+"="*80+"\n\n")
            f.write("DATASET:\n")
            f.write(f"  Total selected  : {config.TOTAL_SAMPLE_SIZE}\n")
            f.write(f"  Per-class quota : {config.SAMPLES_PER_CLASS}–{config.SAMPLES_PER_CLASS+1}\n")
            f.write(f"  Split           : {config.TRAIN_SAMPLES} train / {config.VAL_SAMPLES} val / {config.TEST_SAMPLES} test\n")
            f.write(f"  Leakage check   : val ∩ test = 0 \n\n")
            f.write("RAG SYSTEMS:\n")
            f.write(f"  CLIP  : {config.CLIP_MODEL} → PCA {config.PCA_DIM}-dim\n")
            f.write(f"  RAG 1 : FAISS IndexFlatIP, TTA×{config.TTA_VIEWS}, k={config.TOP_K_SIMILAR}\n")
            f.write(f"  VLM   : BLIP-2 w={config.VLM_RERANK_WEIGHT:.2f}\n")
            f.write(f"  RAG 2 : RF+GBM+FAISS on SmartFarming CSV\n\n")
            f.write("AGENT 1 — OVERALL METRICS:\n"+"-"*50+"\n")
            for k, v in [("Accuracy",metrics["accuracy"]),("F1 Macro",metrics["f1_score_macro"]),
                         ("F1 Weighted",metrics["f1_score_weighted"]),("Prec Macro",metrics["precision_macro"]),
                         ("Prec Weighted",metrics["precision_weighted"]),("ROC-AUC",metrics["roc_auc_macro"]),
                         ("CO2 kg",metrics["co2_emissions_kg"]),("N test",float(metrics["num_eval_samples"]))]:
                f.write(f"  {k:<26}: {v:.6f}\n")
            f.write("\nPER-CLASS METRICS:\n"+"-"*75+"\n")
            f.write(f"{'Class':<45} {'Prec':>8} {'Recall':>8} {'F1':>8} {'N':>6}\n"+"-"*75+"\n")
            for cn in self.eval_label_names:
                m = metrics["per_class_metrics"].get(cn,{})
                f.write(f"{cn:<45} {m.get('precision',0):>8.4f} {m.get('recall',0):>8.4f} "
                        f"{m.get('f1_score',0):>8.4f} {m.get('support',0):>6}\n")
            f.write("\n"+"="*80+"\n  AGENT 2 — CROP RECOMMENDATIONS\n"+"="*80+"\n")
            f.write(f"  {'Rank':<5} {'Crop':<20} {'Score':>8} {'Conf%':>7} {'Season':<12} Soil\n"+"  "+"-"*68+"\n")
            for i, r in enumerate(recs):
                f.write(f"  {i+1:<5} {r.crop:<20} {r.score:>8.4f} {r.confidence_pct:>6.1f}% "
                        f"{r.season:<12} {r.soil_match}\n")
                f.write(f"       → {r.reasoning}\n")
            f.write("\n"+"="*80+"\n  END OF REPORT\n"+"="*80+"\n")
        print(f" Report → {path}"); return path

    def save_metrics_json(self, metrics) -> str:
        path = os.path.join(config.OUTPUT_DIR, "metrics_summary.json")
        summary = {
            "accuracy":           round(metrics["accuracy"],6),
            "f1_score_macro":     round(metrics["f1_score_macro"],6),
            "f1_score_weighted":  round(metrics["f1_score_weighted"],6),
            "precision_macro":    round(metrics["precision_macro"],6),
            "precision_weighted": round(metrics["precision_weighted"],6),
            "roc_auc_macro":      round(metrics["roc_auc_macro"],6),
            "co2_emissions_kg":   round(metrics["co2_emissions_kg"],8),
            "num_eval_samples":   metrics["num_eval_samples"],
            "per_class": {c:{k:(round(v,4) if isinstance(v,float) else v)
                            for k,v in metrics["per_class_metrics"][c].items()}
                         for c in self.eval_label_names},
        }
        with open(path,"w") as f: json.dump(summary, f, indent=2)
        print(f" Metrics JSON → {path}"); return path

    def save_predictions_csv(self, metrics) -> str:
        path = os.path.join(config.OUTPUT_DIR, "predictions.csv")
        pd.DataFrame(metrics["predictions"]).to_csv(path, index=False)
        print(f" Predictions CSV → {path}"); return path

    def save_classification_report_csv(self, metrics) -> str:
        path = os.path.join(config.OUTPUT_DIR, "classification_report.csv")
        rows = []
        for cls_key, vals in metrics["classification_report"].items():
            if isinstance(vals, dict):
                rows.append({"class":cls_key,
                             **{k:(round(v,4) if isinstance(v,float) else v)
                                for k,v in vals.items()}})
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f" Classification report CSV → {path}"); return path

    def save_confusion_matrix_csv(self, metrics) -> str:
        path = os.path.join(config.OUTPUT_DIR, "confusion_matrix.csv")
        pd.DataFrame(metrics["confusion_matrix"],
                     index=self.short, columns=self.short).to_csv(path)
        print(f" Confusion matrix CSV → {path}"); return path

# RUN EVALUATION + SAVE EVERYTHING

print("\n"+"="*60)
print(f" LOADING TEST SPLIT")
print("="*60+"\n")

test_images = processor.load_split("test", filter_classes=ALLOWED_LABEL_INDICES)
print(f"   Test images: {len(test_images):,}")

evaluator = ModelEvaluator(
    orchestrator, processor, label_names,
    eval_class_indices=EVAL_CLASS_INDICES,
    eval_label_names=EVAL_LABEL_NAMES)

metrics = evaluator.evaluate_on_test_set(test_images)

print("\n Saving all results...")
evaluator.generate_all_plots(metrics, location=config.DEFAULT_LOCATION)
evaluator.save_metrics_report(metrics)
evaluator.save_metrics_json(metrics)
evaluator.save_predictions_csv(metrics)
evaluator.save_classification_report_csv(metrics)
evaluator.save_confusion_matrix_csv(metrics)

print("\n"+"="*60)
print(" AGENT 2 — CROP RECOMMENDATIONS")
print("="*60)

demo_queries = [
    {"location":"West Bengal, India (Rice Belt)","temperature":28.0,"humidity":82.0,
     "rainfall":1600.0,"ph":6.0,"nitrogen":80.0,"phosphorus":40.0,"potassium":40.0,
     "soil_type":"alluvial","season":"kharif"},
    {"location":"Punjab, India (Wheat Belt)","temperature":14.0,"humidity":55.0,
     "rainfall":350.0,"ph":7.5,"nitrogen":120.0,"phosphorus":60.0,"potassium":40.0,
     "soil_type":"loam","season":"rabi"},
    {"location":"Kerala, India (Tropical Coast)","temperature":30.0,"humidity":90.0,
     "rainfall":3000.0,"ph":5.5,"nitrogen":50.0,"phosphorus":30.0,"potassium":50.0,
     "soil_type":"laterite","season":"kharif"},
]

demo_rows = []
for q in demo_queries:
    loc = q.pop("location")
    res = crop_agent.run(location=loc, **q)
    print(f"\n {loc}")
    for r in res["top_recommendations"]:
        print(f"   {r['rank']}. {r['crop']:<20} score={r['score']:.4f} "
              f"conf={r['confidence_pct']:.1f}% {r['season']}")
        demo_rows.append({"location":loc, **r})

demo_csv = os.path.join(config.OUTPUT_DIR, "crop_recommendations_demo.csv")
pd.DataFrame(demo_rows).to_csv(demo_csv, index=False)
print(f"\n Crop demo CSV → {demo_csv}")

print("\n"+"="*60+" OUTPUT FILES  "+"="*60)
from pathlib import Path
all_files  = sorted(Path(config.OUTPUT_DIR).rglob("*.*"))
total_kb   = 0
for fp in all_files:
    kb = fp.stat().st_size / 1024; total_kb += kb
    rel = fp.relative_to(config.OUTPUT_DIR)
    print(f"   {str(rel):<65} {kb:>8.1f} KB")
print(f"\n   {'TOTAL':<65} {total_kb:>8.1f} KB")

print("\n"+"="*60)
print(" PIPELINE COMPLETE!  [Enhanced BS/LB Focus]")
print(f"     Dataset  : {config.TOTAL_SAMPLE_SIZE} images (50% balanced)")
print(f"     Split    : {config.TRAIN_SAMPLES} train / {config.VAL_SAMPLES} val / {config.TEST_SAMPLES} test")
print(f"     Accuracy : {metrics['accuracy']:.4f}")
print(f"     F1 Macro : {metrics['f1_score_macro']:.4f}")
print(f"     ROC-AUC  : {metrics['roc_auc_macro']:.4f}")
print("="*60)